In [ ]:
from __future__ import division
import numpy as np
import scipy
import time
import matplotlib.pyplot as plt
plt.rcParams.update( plt.rcParamsDefault )
import warnings
warnings.filterwarnings( 'ignore' )
np.random.seed(1234)
plt.rcParams.update( { 'font.size' : 12 } ) 
%matplotlib inline
%load_ext autoreload
%autoreload 

In [ ]:
"""Creating image storage path and folder"""
import os
relative_path_to_new_folder = "../Images"
os.makedirs( relative_path_to_new_folder, exist_ok = True )
image_folder_path = relative_path_to_new_folder + "/Hybrid_optimization_images"
if not os.path.isdir( image_folder_path ):
    os.makedirs( image_folder_path )

# Theoretical framework

## Entropy regularized formulation

The primal entropy regularized formulation is given by:
$$
OT_{\epsilon}(\alpha,\beta) = min_{\pi \in \mathcal{U}(\alpha,\beta)} \langle C,\pi \rangle +\epsilon KL\left(\pi\|\alpha \otimes \beta\right)\ ,
$$
where
$\ 
KL(\pi\|\alpha \otimes \beta) 
\ $ is the KL-divergence and $\ \mathcal{U}(\alpha,\beta)=\{\pi: \pi\mathbb{1}_{m}=\alpha\ , \pi^{T}\mathbb{1}_{n}=\beta\}\ .$

The optimal coupling $\pi^{*}$ has the following form :
$$
\pi^{*} = \alpha \odot \Delta(u^{*})K \Delta(v^{*})\odot \beta\ ,\ \Delta: diag: \mathbb{R}^{n} \mapsto M_{n}\left(\mathbb{R}\right)
\ .$$

## I. Sinkhorn
The Sinkhorn updates is given by the following alternative projections
$$
u^{(t+1)}  \leftarrow \frac{1}{K\left(v^{(t)}\odot \beta\right)}\ ,\ 
v^{(t+1)}  \leftarrow \frac{1}{K^{T}\left(u^{(t+1)}\odot \alpha\right)}\ , 
$$
where 
$K = \exp\left(-\frac{C}{\epsilon}\right)\in M_{n,\ m}(\mathbb{R}),\ \alpha \in \mathbb{R}^{n},\ \beta \in \mathbb{R}^{m}\ ,\ u\in \mathbb{R}^{n},\ v\in \mathbb{R}^{m}\ $ and $ \ (u^{(0)},v^{(0)})=(u,v)\ .$

Note that, the denominators  $K\left(v^{(t)}\odot \beta\right)$ and $K^{T}\left(u^{(t+1)}\odot \alpha\right)$ are sum of exponentials which is prone to both overflow and underflow when $\varepsilon$ is small, causing the algorithm to terminate without convergence.

## II. Log-domain Sinkhorn

By log-domain we mean the log-exp regularization, that is, we subtract the maximum exponent from each of the exponents in a sum of exponentials ensuring that the maximum value of exponentials in the sum is one which also ensures that the sum is always positive. This immunes the sum of exponentials to overflow and underflow of values.

The log-exp regularized update of the Sinkhorn algorithm at iteration $t+1$ is given by
$$
m^{g}_{i}\left(g^{(t)}\right)\leftarrow \max_{j}\left( g^{(t)}_{j} - C_{ij} \right)\ ,\ \forall\  i = 1,\dots,n\ ,
$$
$$
f^{(t+1)}_{i}\leftarrow -\varepsilon \log\left(\sum_{j=1}^{m}\exp\left(\frac{\left( g_{j}^{(t)} - C_{ij} - m^{g}_{i}(g^{(t)})\right)}{\varepsilon}\right)\beta_{j}\right)-m^{g}_{i}\left(g^{(t)}\right)\ ,\ \forall\  i=1,\dots,n\ ,
$$
$$
m^{f}_{j}\left(f^{(t+1)}\right)\leftarrow \max_{i}\left( f^{(t+1)}_{i} - C_{ij} \right)\ ,\ \forall\   j=1,\dots,m\
 ,
$$
$$
g^{(t+1)}_{j}\leftarrow -\varepsilon \log\left(\sum_{i=1}^{n}\exp\left(\frac{\left( f_{i}^{(t+1)} - C_{ij} - m^{f}_{j}(f^{(t+1)})\right)}{\varepsilon}\right)\alpha_{i}\right) - m^{f}_{j}\left(f^{(t+1)}\right)\ ,\ \forall\  j=1,\dots,m\ ,
$$
where $C \in M_{n,\ m}(\mathbb{R}),\ \varepsilon >0,\ $ $\alpha \in \mathbb{R}^{n},\ $ $\beta \in \mathbb{R}^{m},\ $
   $f \in \mathbb{R}^{n},\ $ $g \in \mathbb{R}^{m}\ $ and $\ (f^{(0)},g^{(0)})=(f,g)\ .$ 

## Dual of Entropic formulation
The dual of the entropic formulation of the OT is given by

\begin{align*}
\max_{(f,g)\in \mathbb{R}^{n}\times\mathbb{R}^{m} }Q_{C,\alpha,\beta,\varepsilon}(f,g):=\langle f,\ \alpha \rangle 
		+ \langle g,\ \beta \rangle
		-\varepsilon 
\left(\langle \exp\left(\frac{f+g-C}{\varepsilon}\right),\ \alpha\otimes \beta\rangle-1\right)\ ,
\end{align*}	
where $\odot$ denotes the elementwise product, $C \in M_{n,\ m}(\mathbb{R}),\ \varepsilon >0,\ $ $\alpha \in \mathbb{R}^{n},\ $ $\beta \in \mathbb{R}^{m},\ $
   $f \in \mathbb{R}^{n},\ $ $g \in \mathbb{R}^{m}\ $ .

## Semi-dual formulation
The semi-dual formulation is obtained by plugging-in one of the Schrodinger-bridge relations between the two optimum potentials in the dual of the entropy regularized formulation.
Therefore, on plugging-in the relation of $g$ with $f$ we have
\begin{align*}
Q^{semi}_{C,\alpha,\beta,\varepsilon}(f) & := \langle f,\ \alpha \rangle 
		+ \langle g(f,C,\alpha,\varepsilon),\ \beta \rangle
		-\varepsilon 
\left(\langle  \exp\left(\frac{f+g(f,C,\alpha,\varepsilon)-C}{\varepsilon}\right),\ \alpha\otimes \beta\rangle-1\right)\\
& = \langle f,\ \alpha \rangle 
		+ \langle g(f,C,\alpha,\varepsilon),\ \beta \rangle
		-\varepsilon 
\left( 1-1\right)\\
& = \langle f,\ \alpha \rangle + \langle g(f,C,\alpha,\varepsilon),\ \beta \rangle\ , 
\end{align*}
where
$g(f,C,\alpha,\varepsilon)_{j} := -\varepsilon\log\left(\sum_{i=1}^{n}\exp\left(\frac{f_{i}-C_{ij}}{\varepsilon}\right)\alpha_{i}\right)\ ,\ \forall j = 1,\dots,m\ .$

Let us look at the gradients and the Hessian of this formulation.

Let 
\begin{align*}\Gamma^{C,\alpha}(f,\varepsilon)_{ij} := \frac{\alpha_{i}\exp\left(\frac{f_{i} - C_{ij}}{\varepsilon}\right)}{\left(\sum_{t=1}^{n}\alpha_{i}\exp\left(\frac{f_{i} - C_{ij}}{\varepsilon}\right)\right)}\ ,\ \forall\ i = 1,\dots, n\ ;\ j = 1,\dots, m\ .
\end{align*}
### Gradients

\begin{align*}
\nabla_{f}Q^{semi}_{C,\alpha,\beta,\varepsilon}(f)_{i} &= \alpha_{i}-\sum_{j=1}^{m}\Gamma^{C,\alpha}(f,\varepsilon)_{ij}\beta_{j}\ ,\ \forall\ i = 1,\dots,n\ .
\end{align*}

### Hessian 

\begin{align*}
\nabla_{f}^{2}Q^{semi}_{C,\alpha,\beta,\varepsilon}(f) = \frac{-1}{\varepsilon}\left(\Delta\left( \left( \Gamma^{C,\alpha}(f,\varepsilon)\odot \beta \right)\mathbb{1}_{m}\right)- \Gamma^{C,\alpha}(f,\varepsilon)\Delta(\beta)\left( \Gamma^{C,\alpha}(f,\varepsilon)\right)^{T}\right) ,
\end{align*}
where $\odot$ is the elementwise multiplication and $\Delta: diag: \mathbb{R}^{n} \mapsto M_{n}(\mathbb{R})\ .$

### Regularization
The log-exp regularization of $g(f,C,\alpha,\varepsilon)$ is the same as that in log-domain Sinkhorn while the regularization of the gradients and the Hessian can be done by regularizing the matrix $\Gamma^{C,\alpha}(f,\varepsilon)$ in the following way
\begin{align*}
\Gamma^{C,\alpha}(f,\varepsilon)_{ij} = \frac{\alpha_{i}\exp\left(\frac{f_{i} - C_{ij} - m^{f}_{j}(f)}{\varepsilon}\right)}{\left( \sum_{t=1}^{n}\alpha_{i}\exp\left(\frac{f_{i} - C_{ij} - m^{f}_{j}(f)}{\varepsilon}\right)\right)}\ ,\ \forall\ i = 1, \dots,n\ ;\ j = 1,\dots,m\ ,
\end{align*}
where $m^{f}_{j}(f) := \max_{i}\left( f_{i} - C_{ij} \right)\ ,\ \forall\ j = 1,\dots,m\ .$ Note that this regularization ensures that there are no overflows or underflow in the denominator in the entries of $\Gamma^{C,\alpha}(f,\varepsilon)$ but this can only immune the numerator to overflows and not underflows when $\varepsilon$ is small.

### Remark
#### Instability in the Hessian for small $\varepsilon$
If we carefully look at the entries of the Hessian, we have
\begin{align*}
H^{C,\alpha, \beta}(f,\varepsilon)_{ii} = \sum_{s=1}^{m}\Gamma^{C,\alpha}(f,\varepsilon)_{is}\left(1 - \Gamma^{C,\alpha}(f,\varepsilon)_{is}\right)\beta_{s}\ ,\ \forall\ i = 1 \dots,n\ ,
\end{align*}
and
\begin{align*}
H^{C,\alpha, \beta}(f,\varepsilon)_{ij} = -\sum_{s=1}^{m}\Gamma^{C,\alpha}(f,\varepsilon)_{is}\Gamma^{C,\alpha}(f,\varepsilon)_{js}\beta_{s}\ ,\ \forall\ i\neq j = 1 \dots,n\ .
\end{align*}
Note that as $\varepsilon \rightarrow 0\ ,$
\begin{align*}
\Gamma^{C,\alpha}(f,\varepsilon)_{is}\rightarrow
\begin{cases} 
\frac{\alpha_{i}}{\left(\sum_{\substack{1 \leq t \leq q \leq n \\ i_1 = i}}\alpha_{i_{t}}\right)} &,\ \text{if } f_{i}-C_{is}=\max_{i}\left(f_{i}-C_{is}\right) \\
0 &,\ \ otherwise\ ,\ \forall\ i = 1,\dots,n\ ;\ s=1,\dots,m\ ,
\end{cases}
\end{align*}
where $q$ is the number of elements in the column $s$ attaining the maximum value.

That is, many entries of the Hessian will be close to zero as $\varepsilon$ decreases. For instance, if $q=1$ for all elements in the row $i$ then the corresponding diagonal element at $(i,i)$ will vanish. This is due to the underflow that occurs in the numerator of the entries $\Gamma^{C,\alpha}(f,\varepsilon)\ .$
This may incur instability when performing damped Newton using exact inversion.


## III. Damped Newton with preconditioning
### Preconditioned inversion of the Hessian 

Let $H^{C,\alpha, \beta}(f,\varepsilon)$ denote the unnormalized Hessian, that is, $H^{C,\alpha, \beta}(f,\varepsilon) := - \varepsilon \nabla^{2}_{f}Q^{semi}_{C,\alpha,\beta,\varepsilon}(f)\ .$

By Perron-Frobenius theorem $\dim\left(\Delta\left(\frac{1}{\sqrt{\alpha}}\right)H^{C,\alpha, \beta}(f,\varepsilon)\Delta\left(\frac{1}{\sqrt{\alpha}}\right)\right) = 1 $ and $\mathbb{1}_{n}\in \ker\left(\Delta\left(\frac{1}{\sqrt{\alpha}}\right)H^{C,\alpha, \beta}(f,\varepsilon)\Delta\left(\frac{1}{\sqrt{\alpha}}\right)\right)\ .$ Therefore, the Hessian is not invertible. Also if we observe the spectrum of the Hessian we find that as $\varepsilon$ decreases more eigenvalues clutter near zero. 

To ensure the stability of inverting the Hessian during damped Newton we precondition it in two stages:
#### i. Null vector preconditioning:
Here we precondition the Hessian with the null vector to make sure it is invertible.
\begin{align*}
    \tilde{H}^{C,\alpha, \beta}(f,\varepsilon)  := \Delta\left(\frac{1}{\sqrt{\alpha}}\right)H^{C,\alpha, \beta}(f,\varepsilon)\Delta\left(\frac{1}{\sqrt{\alpha}}\right) + \tau \mathbb{1}_{n}\mathbb{1}_{n}^{T}\ .
\end{align*}
#### ii. True preconditioning:
Here we move $k\leq n$ non-zero eigenvalues that are close to 0 to 1 which ensures stability of the inversion. The preconditioning matrix is given by
\begin{align*}
P := I_{n} + \sum_{i=1}^{k}\left(\frac{1}{\sqrt{\lambda}}-1\right)y_{i}y_{i}^{T}\ ,
\end{align*}
where $y_{i}\in \ker\left(\Delta\left(\frac{1}{\sqrt{\alpha}}\right)\tilde{H}^{C,\alpha, \beta}(f,\varepsilon)\Delta\left(\frac{1}{\sqrt{\alpha}}\right)-\lambda_{i}I_{n}\right)\ ,\ \langle y_{i}, y_{j}\rangle = \delta_{ij}\ $
such that
\begin{align*}
P\Delta\left(\frac{1}{\sqrt{\alpha}}\right)\tilde{H}^{C,\alpha, \beta}(f,\varepsilon)\Delta\left(\frac{1}{\sqrt{\alpha}}\right)Py_{i} = y_{i}\ ,\ \forall\ i = 1,\dots,k\ ,
\end{align*}
where $\delta_{ij}$ is the Kronecker delta.

For inversion we either use exact or iterative methods of inversion such as conjugate gradient or GMRES. If $p$ is the ascent direction, then the original ascent direction is given by  $p \leftarrow\frac{1}{\sqrt{\alpha}} \odot p\ .$
### Line search

For $\tau = 1,\ c,\ \rho \ \in\  (0,1)\ ,$

while $Q^{semi}_{C,\alpha,\beta,\varepsilon}( f + \tau p ) < Q^{semi}_{C,\alpha,\beta,\varepsilon}( f ) + \tau c \langle p, \nabla_{f}Q^{semi}_{C,\alpha,\beta,\varepsilon}(f) \rangle: $
$$
\tau \leftarrow \rho \tau\ ,
$$
where $\tau$ is the update step size, $c$ is the sufficient increase parameter and $\rho$ is the damping factor.

When we have such $p$ and $\tau\ ,$ we do the update $f\leftarrow f + \tau p\ .$

### Remark:
####  Role of the sufficient increase parameter $c$ and damping factor $\rho$
The parameter $c$ controls the slope of the line in the Armijo condition. A large value of $c$ indicates a strict condition while a small value indicates a lenient condition.
Small $c$ ensure that less time is spent in finding the correct step size while a large value of $c$ indicates that more reductions of the initial step size is required.
On the other hand, a small to moderate damping factor indicates aggressive damping to meet the condition.
So a small to moderate $c$ and $\rho$ therefore ensures less time spent in finding the appropriate step size that meets the Armijo condition. 


# Code

## Code 1: Sinkhorn

In [ ]:
class sinkhorn:
  def __init__( self, K, a, b, u, v, epsilon ):
    """
                                                                                          
      Parameters: 
      -----------
      K : ndarray, shape (n,m)
          The Gibb's kernel.
      a : ndarray, shape (n,)
          The probability histogram of the sample of size n.
      b : ndarray, shape (m,)
          The probability histogram of the sample of size m.
      u : ndarray, shape (n,)
          The initial u.
      v : ndarray, shape (m,)
          The initial v.
      epsilon : float
                The regularization parameter.
    """
    self.K = K
    self.a = a
    self.b = b
    self.u = u
    self.v = v
    self.epsilon = epsilon
    self.errors = []
    self.objective_values = []
    
  def _objectivefunction( self ):
    """
    
      Returns:
      --------
      Q(f,g)  : float
                The value of objective function obtained by evaluating the formula Q(f,g) = < f, a > + < g, b > - epsilon * ( < a * u, Kv * b > - 1 ),
                where u = exp( f/epsilon ), v = exp( g/epsilon ). 
    """
    f = np.log( self.u ) * self.epsilon
    g = np.log( self.v ) * self.epsilon
    target = np.dot( f, self.a ) + np.dot( g, self.b )
    penalization = -self.epsilon * ( np.dot( self.a * np.exp( f/self.epsilon ).T, np.dot( self.K, np.exp( g/self.epsilon ) * self.b ) ) - 1 )
    return target + penalization

  def _optimize( self, tol = 1e-12, max_iterations = 1000 ):
    """

      Parameters:
      -----------
      tol : float
            The toleranc for the error. Defaults to 1e-12.
      max_iterations  : int
                        The maximum iteration for the optimization algorithm. Defaults to 1000.

      Returns:
      --------
      Returns a dictionary where the keys are strings and the values are ndarrays or list.
      The following are the keys of the dictionary and the descriptions of their values:

      potential_f : ndarray, shape (n,)
                    The optimal Kantorovich potential f.
      potential_g : ndarray, shape (m,)
                    The optimal Kantorovich potential g.
      errors  : list
                The list of errors observed when checking conservation of mass .
      objective_values  : list
                          The list of objective values observed after each ascent update.
    """
    i = 1
    while True :
      # Projection 1:
      self.u = 1/ np.dot( self.K, self.v * self.b )# Shape:(n,)
      # Estimating marginal b:
      b_hat = self.b * self.v * np.dot( self.K.T, self.u * self.a )
      # Projection 2:
      self.v = 1 / np.dot( self.K.T, self.u * self.a )# Shape:(m,)
      # Estimating marginal a:
      a_hat = self.a * self.u * np.dot( self.K, self.v * self.b )
      # Check conservation of mass:
      self.errors.append(  np.linalg.norm( b_hat - self.b )
                           +
                           np.linalg.norm( a_hat - self.a )
                        )
      self.objective_values.append( self._objectivefunction() )
      condition = ( self.errors[-1] > tol )
      if i < max_iterations and condition :
          i += 1
      elif np.isnan( self.errors[-1] ):
        print( "Terminating at iteration: ", i,", due to underflow and overflow of values in the denominator of the projections." )
        break
      else:
        print( "Terminating after iteration: ", i  )
        break   
    # end while
    # Computing the potentials:
    f = self.epsilon * np.log( self.a * self.u )
    g = self.epsilon * np.log( self.b * self.v )
    # Computing the coupling matrix:
    P = self.a[:,None] * np.exp( ( f[:,None] + g[None,:] + np.log( self.K ) )/self.epsilon ) * self.b[None,:]
    return {
      'potential_f'       : f,
      'potential_g'       : g,
      'coupling_matrix'   : P,
      'errors'            : self.errors,
      'objective_values'  : self.objective_values
      }


## Code 2: Log-domain Sinkhorn

In [ ]:
class log_domainSinkhorn:
    def __init__( self, a, b, C, epsilon ):
        """
        
            Parameters:
            -----------
            C   :   ndarray, shape (n,m) 
                    It is the cost matrix between the points sampled from the point clouds.
            a   :   ndarray, shape (n,)
                    The probability histogram of the sample of size n.
            b   :   ndarray, shape (m,)
                    The probability histogram of the sample of size m.
            epsilon     :   float
                            The regularization parameter.
        """
        self.a = a
        self.b = b
        self.C = C
        self.epsilon = epsilon
        self.errors = []
        self.objective_values = []
    
    def _objectivefunction( self ):
        """
        
            Returns:
            --------
            Q(f,g)  :   float
                        The value of objective function obtained by evaluating the formula  Q(f,g) = < f, a > + < g, b > - epsilon * ( < a * u, Kv * b > - 1 ),
                        where u = exp( f/epsilon ), v = exp( g/epsilon ). 
        """
        K = np.exp( -self.C/self.epsilon )
        target = np.dot( self.f, self.a ) + np.dot( self.g, self.b )
        penalization = -self.epsilon * ( np.dot( self.a * np.exp( self.f/self.epsilon ).T, np.dot( K, np.exp( self.g/self.epsilon ) * self.b ) ) - 1 )
        return target + penalization
    
    def _f( self, g ):
        """

            Here we compute the value of the potential f by using its Schrodinger-bridge relation with the potential g: 
            f = - epsilon * log( ( b * exp( ( g - C )/epsilon ) )1_{m} ).

            Parameters:
            -----------
            g   :   ndarray, shape (m,)
                    The input Kantorovich potential g.
            Returns:
            --------
            ndarray, shape (n,)
            The value of potential f.
        """
        f = - self.epsilon * np.log( np.sum( self.b[None,:] * np.exp( ( g[None,:] - self.C ) /self.epsilon ), axis = 1 ) )
        return f# Shape: (n,)
    
    def _logexp_f( self, g ):
        """

            Here we incorporate the log-exp regularization method to the computation of the potential f by using its Schrodinger-bridge relation with the potential g: 
            f = - epsilon * log( ( b * exp( ( g - C - max_g )/epsilon ) )1_{m} ) - max_g,
            where max_g is the maximum value along each row of g - C.

            Parameters:
            -----------
            g   :   ndarray, shape (m,)
                    The input Kantorovich potential g.

            Returns:
            --------
            ndarray, shape (n,)
            The log-exp regularized value of potential f.
        """
        max_g = np.max(  g[None,:] - self.C, axis = 1 )
        f = - self.epsilon * np.log( np.sum( self.b[None,:] * np.exp( ( g[None,:] - self.C - max_g[:,None] )/self.epsilon ), axis = 1 ) ) - max_g
        return f# Shape: (n,)

    def _g( self, f ):
        """

            Here we compute the value of the potential g by using its Schrodinger-bridge relation with the potential f: 
            g = - epsilon * log( 1_{n}^{T} ( a * exp( ( f - C )/epsilon ) ) ).

            Parameters:
            -----------
            f   :   ndarray, shape (n,)
                    The input Kantorovich potential f.
            Returns:
            --------
            ndarray, shape (m,)
            The value of potential g.
        """
        g = - self.epsilon * np.log( np.sum( self.a[:,None] * np.exp( ( f[:,None] - self.C )/self.epsilon ), axis = 0 ) )
        return g# Shape: (m,)
    
    def _logexp_g( self, f ):
        """

            Here we incorporate the log-exp regularization method to the computation of the potential g by using its Schrodinger-bridge relation with the potential f: 
            g = - epsilon * log( 1_{n}^{T} ( a * exp( ( f - C - max_f )/epsilon ) ) ) - max_f,
            where max_f is the maximum along each column of f - C.
            
            Parameters:
            -----------
            f   :   ndarray, shape (n,)
                    The input Kantorovich potential f.

            Returns:
            --------
            ndarray, shape (m,)
            The log-exp regularized value of potential g.

        """
        max_f = np.max( f[:,None] - self.C , axis = 0 )
        g =  - self.epsilon * np.log( np.sum( self.a[:,None] * np.exp( ( f[:,None] - self.C - max_f[None,:] )/self.epsilon ), axis = 0 ) )  - max_f
        return g# Shape: (m,)
    
    def _optimize( self, tol = 1e-12, max_iterations = 500 ):     
        """
        
            Parameters:
            -----------
            tol :   float
                    The tolerance for the error. Defaults to 1e-12.
            max_iterations  :   int
                                The maximum iteration for the optimization algorithm. Defaults to 500.
            Returns:
            --------
            Returns a dictionary where the keys are strings and the values are ndarrays or list.
            The following are the keys of the dictionary and the descriptions of their values:

            potential_f     :   ndarray, shape (n,)
                                The optimal Kantorovich potential f.
            potential_g     :   ndarray, shape (m,)
                                The optimal Kantorovich potential g.
            errors  :   list
                        The list of errors observed when checking conservation of mass .
            objective_values    :   list
                                    The list of objective values observed after each ascent update.
        """
        i = 1
        self.f = self.a
        while True:
            self.g = self._logexp_g( self.f )# Shape: (m,)
            self.f = self._logexp_f( self.g )# Shape: (n,)
            # Computing the coupling matrix:
            P = self.a[:,None] * np.exp( ( self.f[:,None] + self.g[None,:]  - self.C )/self.epsilon ) * self.b[None,:]# Shape: (n,m), line (*)
            # Check conservation of mass: ||P1_m - a||_1 + ||P^T 1_n - b||_1:
            self.errors.append(  np.linalg.norm( np.sum( P, axis = 1 ) - self.a, ord = 1 )
                                 +
                                 np.linalg.norm( np.sum( P, axis = 0 ) - self.b, ord = 1 )
                             )
            self.objective_values.append( self._objectivefunction() )
            condition = ( self.errors[-1] > tol )
            if i + 1 < max_iterations and condition:
                i += 1
            else:
                print( "Terminating after iteration: ", i  )
                break
        # end while
        # Change of convention because of line (*)
        self.f = self.f + self.epsilon * np.log( self.a )
        self.g = self.g + self.epsilon * np.log( self.b )
        return {
            'potential_f'       : self.f,
            'potential_g'       : self.g, 
            'coupling_matrix'   : P,
            'errors'            : self.errors, 
            'objective_values'  : self.objective_values,                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 
        }

## Code 3: Damped Newton with preconditioning in the semi-dual framework

In [ ]:
class semi_dual_damped_Newton_with_preconditioning:
    def __init__( self, C, a, b, f, epsilon, rho, c, null_vector, precond_vectors ):
        """

            Parameters:
            ----------- 
            C   :   ndarray, shape (n,m) 
                    It is the cost matrix between the points sampled from the point clouds.
            a   :   ndarray, shape (n,)
                    The probability histogram of the sample of size n.
            b   :   ndarray, shape (m,)
                    The probability histogram of the sample of size m.
            f   :   ndarray, shape (n,) 
                    The initial Kantorovich potential f.
            rho     :   float
                        Damping factor for the line search ascent step size.
            c   :   float
                    Sufficient increase parameter in the Armijo condition.
            epsilon     :   float
                            The regularization parameter.
            null_vector     :   ndarray, shape (n,)
                                The null vector of the Hessian to be used for null vector preconditioning.
            precond_vectors     :   list of ndarrays, shape (n,)
                                    The stack of preconditioning vectors obtained from the Hessian at the optimum obtained from the algorithm without any preconditioning,
                                    that is, semi-dual damped Newton with only null vector preconditioning and exact inversion.
        """
        self.C = C
        self.a = a
        self.b = b
        self.f = f
        self.epsilon = epsilon
        self.rho = rho           
        self.c = c
        self.null_vector = null_vector
        self.precond_vectors = precond_vectors
        self.timings_each_step = []
        self.inversion_timings = []
        self.tau_list = []
        self.errors = []
        self.objective_values = [] 
        self.backtracking_info = []
        
    def _objectivefunction( self, f, g ) :
        """ 

            Parameters:
            -----------
            f   :   ndarray, shape (n,)
                    The input Kantorovich potential f.
            g   :   ndarray, shape (m,)
                    The input Kantorovich potential g.
                    
            Returns: 
            --------
            Q_semi(f)   :   float
                            The value of semi-dual objective function obtained by evaluating Q_semi(f) = < f, a > + < g( f, C, a, epsilon ), b >,
                            where g( f, C, a, epsilon ) denotes the value of Kantorovich potential g evaluated using the Schrodinger-bridge equations between f and g.
         """
        Q_semi = np.dot( f, self.a ) + np.dot( g, self.b )
        return Q_semi
    
    def _g( self, f ):
        """

            Here we compute the value of the potential g by using its Schrodinger-bridge relation with the potential f: 
            g = - epsilon * log( 1_{n}^{T} ( a * exp( ( f - C )/epsilon ) ) ).

            Parameters:
            -----------
            f   :   ndarray, shape (n,)
                    The input Kantorovich potential f.
            Returns:
            --------
            ndarray, shape (m,)
            The value of potential g.
        """
        g = - self.epsilon * np.log( np.sum( self.a[:,None] * np.exp( ( f[:,None] - self.C )/self.epsilon ), axis = 0 ) )
        return g# Shape: (m,)
    
    def _logexp_g( self, f ):
        """

            Here we incorporate the log-exp regularization method to the computation of the potential g by using its Schrodinger-bridge relation with the potential f: 
            g = - epsilon * log( 1_{n}^{T} ( a * exp( ( f - C - max_f )/epsilon ) ) ) - max_f,
            where max_f is the maximum along each column of f - C.
            
            Parameters:
            -----------
            f   :   ndarray, shape (n,)
                    The input Kantorovich potential f.

            Returns:
            --------
            ndarray, shape (m,)
            The log-exp regularized value of potential g.

        """
        max_f = np.max( f[:,None] - self.C , axis = 0 )
        g =  - self.epsilon * np.log( np.sum( self.a[:,None] * np.exp( ( f[:,None] - self.C - max_f[None,:] )/self.epsilon ), axis = 0 ) )  - max_f
        return g# Shape: (m,)
            
    def _backtracking( self, tau, p, slope ):
        """

            Here we use the Armijo condition to decide the ascent step length for updating the potentials towards the ascent direction. 
            Parameters:
            -----------
            tau     :   float  
                        The ascent step size.
            p   :   ndarray, shape (n,)
                    The ascent direction.
            slope   :   float
                        It is the inner product of the gradient and p.
            Returns:
            --------
            tau     :   float
                        The updated ascent step size.
        """
        f_current = self.f# Shape: (n,)
        g_current = self._logexp_g( self.f )# Shape: (m,)
        reduction_count = 0     
        while True:
            f_updated = self.f + tau * p# Shape: (n,)
            g_updated = self._logexp_g( f_updated )# Shape: (m,)
            # Armijo condition:
            condition = self._objectivefunction( f_updated, g_updated ) < self._objectivefunction( f_current, g_current ) + self.c * tau * slope
            if condition or np.isnan( self._objectivefunction( f_updated, g_updated ) ):
                tau = self.rho * tau
                reduction_count += 1
            else:
                break
        # end while
        return tau, reduction_count
    
    def _preconditioned_inversion( self, rtol, atol, inversion_method, max_inversions, show_inversion_timings, show_condition_number ):    
        """

            Parameters:
            -----------
            inversion_method    :   str
                                    The method of inversion to be used for the Hessian. The following are the options:
                                    - "exact" : Exact inversion.
                                    - "cg" : Using conjugate gradient for inversion.
                                    - "GMRES" : Using GMRES for inversion.
            max_inversions  :   int
                                The number of iterative inversions to be performed to obtain the inverse of the Hessian followed by obtaining the ascent direction.
            rtol    :   float
                        The value of relative tolerance which is a hyperparameter to the iterative inversion algorithm, here it is conjugate gradient (CG) or GMRES.
            atol    :   float
                        The value of absolute tolerance which is a hyperparameter to the iterative inversion algorithm, here it is conjugate gradient (CG) or GMRES.    
            show_inversion_timings  :   bool
                                        Indicator to print time stamps at different steps of preconditioned inversion of the Hessian.       
            show_condition_number   :   bool
                                        Indicator to print the condition number of the Hessian matrix.


            Returns:
            --------
            Returns a tuple containing the optimal ascent direction vector p and the recorded timings of various steps of the algorithm. 
            The following are their descriptions:

            p   :   ndarray, shape: (n,)
                    The optimal ascent direction vector.
            timings     :   list
                            The list of timestamps recorded.
        """

        def printing_time( statements ):
            """
                Function to enable the option to print time stamps at different steps.
                Parameters:
                -----------
                statements  :   List of strings to be printed in a line.
            """
            if show_inversion_timings:
                print( " ".join( statements ) )
                
        timings = []
        if show_inversion_timings or show_condition_number or inversion_method in [ 'cg', 'GMRES' ]:
            print( "|"+"-" * 75 )
        start = time.time()
        # Record list of unwinding transformations on final result:
        unwinding_transformations = []
        # Construct modified Hessian:  
        diag = 1/np.sqrt( self.a )          
        self.modified_Hessian = diag[:,None] * self.Hessian * diag[None,:]
        # Dummy variable to work on:
        matrix = self.modified_Hessian
        # Preconditioning along null vector:
        vector = self.null_vector# Shape: (n,)
        vector = vector/np.linalg.norm( vector )
        vector_E = vector
        # Transforming the gradient by conjugation:
        gradient = diag[:,None] * self.gradient_f[:,None]
        # Unwinding transformation to obtain the original direction vector i.e., without any conjugation:
        unwinding_transformations.append( lambda x : diag[:,None] * x )
        end = time.time()
        # Record timings:
        interval = 1e3 * ( end - start )
        timings.append( interval )
        printing_time( [ "|--|- Time required for initial preconditioning: ", str( np.round( interval, 5 ) ), "ms" ])
        # Conditioning with other vectors
        #  Naming conventions:
        #  y = Preconditioning vectors as a numpy matrix n by k
        #  matrix = our matrix A to precondition
        #  We only form the data y and z such that
        #  P = id + z*y.T
        start0 = time.time()
        y = np.array( self.precond_vectors).T # Matrix of size n by k
        # Compute eigenvalues:
        Ay = np.dot( matrix, y )
        eigenvalues = np.sum( y * Ay, axis = 0 )
        # Compute data for P = id + y*diag(values)*y.T:
        values = ( 1/np.sqrt(eigenvalues) - 1 )# Vector of size k
        z = y * values[None,:]
        end = time.time()
        # Record timings:
        interval = 1e3 * ( end - start0 )
        timings.append( interval )
        printing_time( [ "|--|- Time required for preconditioning matrix formation: ", str( np.round( interval, 5 ) ), "ms" ] )
        # Changing A=matrix to PAP:
        start2 = time.time()
        def _apply_P( vector ):
            """

                Function mapping v to Pv, where P = Id + z*y.T.

                Parameters:
                -----------
                vector  :   ndarray, shape: (n,)

                Returns:
                --------
                ndarray, shape: (n,)  
                vector obtained from the map Pv.
            """
            return  vector + z @ ( y.T @ vector ) 
        def _preconditioned_map( vector ):
            """

                Function mapping v to P(A+E)Pv,
                    where   P = P = Id + z*y.T
                            A   :   ndarray, shape: (n,n)                                            
                            E   :   ndarray, shape: (n,n)
                                    outer product of the null vector with itself.       
                Parameters:
                -----------
                    vector  :   ndarray, shape: (n,)
                
                Returns:
                --------
                ndarray, shape: (n,)
                The preconditioned input vector. 
            """
            vector = _apply_P( vector ) 
            vector = np.dot( matrix, vector )  + vector_E * np.dot( vector_E, vector )
            vector = _apply_P( vector ) 
            return vector
        # Apply P
        # At beginning on gradient
        # At the end 
        # Preconditioning the gradient:
        gradient = _apply_P( gradient )
        # The transformation to precondition the direction vector:
        unwinding_transformations.append( lambda x : _apply_P(x) )
        end = time.time()
        # Record timings:
        interval = 1e3 * ( end - start2 )
        timings.append( interval )
        printing_time( [ "|--|- Time required for changing A to PAP: ", str( np.round( interval, 5 ) ), "ms" ] )
        #
        # Solve either iteratively using CG or exactly:
        start3 = time.time()
        if inversion_method != "exact":
            ## Inversion counter:
            inversion_count = [0]
            ## Function to increment counter:
            callback = lambda x: inversion_count.__setitem__(0, inversion_count[0] + 1)
            ## Print condition number:
            if show_condition_number:
                eig, v = np.linalg.eigh( matrix )
                sorting_indices = np.argsort( eig )
                eig = eig[ sorting_indices ]
                v = v[ :, sorting_indices ]
                print( "|--|- Observing eigenvalues and condition number:" )
                print( "|--|- |- List of smallest 10 eigenvalues: [", ", ".join( list( map( str, eig[ : 10 ] ) ) ), "]" )
                print( "|--|- |- List of largest  10 eigenvalues: [", ", ".join( list( map( str, eig[ - 10 : ]  ) ) ), "]" ) 
                print( "|--|- |- Conition number of the Hessian: ", np.linalg.cond( matrix ) )
            self.m  = matrix
            A = scipy.sparse.linalg.LinearOperator( ( self.m.shape[0], self.m.shape[1] ), matvec = _preconditioned_map ) 
            if inversion_method == 'cg':
                print( "|--|- Inverting using conjugate gradient:" )
                inverse, exit_code = scipy.sparse.linalg.cg(    A,
                                                                gradient, 
                                                                x0 = gradient, 
                                                                maxiter = max_inversions, 
                                                                rtol = rtol, 
                                                                atol = atol,
                                                                callback = callback 
                                                            )
                    # print( "  --- CG exit code: ", exit_code)
            else:
                print( "|--|- Inverting using GMRES:" )
                inverse, exit_code = scipy.sparse.linalg.gmres(     A,
                                                                    gradient, 
                                                                    x0 = gradient, 
                                                                    maxiter = max_inversions, 
                                                                    rtol = rtol, 
                                                                    atol = atol,
                                                                    callback = callback 
                                                                )
                # print( "  --- GMRES exit code: ", exit_code)
            p_k = self.epsilon * inverse
            p_k = p_k.reshape( ( p_k.shape[0], 1 ) )# For some reason, this outputs (n,) and the next line outputs (n,1)
            if exit_code == 0:
                inversion_convergence = 'yes'
            else:
                inversion_convergence = 'no'
            print( "|--|- |- Number of iterative inversions: ", inversion_count[0] )
            print( "|--|- |- Convergence status: ", inversion_convergence )
        else:
            # Preconditioning along null vector:                                     
            vector = vector.reshape( ( len(vector), 1 ) )
            matrix = matrix + np.dot( vector, vector.T )     
            # True Preconditioning for exact inverse: 
            B = np.dot( Ay, z.T )
            C = z @ np.dot( y.T, Ay ) @ z.T
            matrix = matrix + B + B.T + C
            self.Hessian_stabilized = - matrix/self.epsilon
             # Print condition number:
            if show_condition_number:
                eig, v = np.linalg.eigh( matrix )
                sorting_indices = np.argsort( eig )
                eig = eig[ sorting_indices ]
                v = v[ :, sorting_indices ]
                print( "|--|- Observing eigenvalues and condition number:" )
                print( "|--|- |- List of smallest 10 eigenvalues: [", ", ".join( list( map( str, eig[ : 10 ] ) ) ), "]" )
                print( "|--|- |- List of largest  10 eigenvalues: [", ", ".join( list( map( str, eig[ - 10 : ]  ) ) ), "]" ) 
                print( "|--|- |- Conition number of the Hessian: ", np.linalg.cond( matrix ) )
            # Performing exact inversion:
            p_k = - np.linalg.solve( self.Hessian_stabilized, gradient )
        end = time.time()
        # Record timings:
        interval = 1e3 * ( end - start3 )
        timings.append( interval )
        printing_time( [ "|--|- Time taken to invert the linear system for p_k: ", str( np.round( interval, 5 )), "ms" ] )
        start4 = time.time()
        # Unwind:
        for transform in unwinding_transformations:
          p_k = transform( p_k )
        end = time.time()
        # Record timings:
        interval = 1e3 * ( end - start4 )
        timings.append( interval )
        printing_time( [ "|--|- Time taken for unwinding: ", str( np.round( interval, 5 ) ), "ms" ] )
        interval = 1e3 * ( end - start )
        timings.append( interval )
        printing_time( [ "|--|- Time taken for the complete code block: ", str( np.round( interval, 5 ) ), "ms"] )
        if show_inversion_timings or show_condition_number or inversion_method in [ 'cg', 'GMRES' ]:
            print( "|"+"-" * 75 )
        return p_k.flatten(), timings
        
    def _optimize( self, tol = 1e-12,  max_iterations = 50, inversion_method = "exact", max_inversions = 30, relative_tol = 1e-5, absolute_tol = 1e-10, show_inversion_timings = False, show_armijo_angle = False, show_condition_number = False, observe_underflows = False ):
        """
        
            Parameters:
            -----------
            tol :   float
                    Tolerance to terminate the algorithm based on the error in estimating the marginals of the coupling.
                    Defaults to 1e-12.
            max_iterations  :   int
                                The maximum number of iterations for the optimization algorithm.
                                Defaults to 50.
            inversion_method    :   str
                                    The method of inversion to be used for the Hessian. The following are the options:
                                    - "exact" : Exact inversion.
                                    - "cg" : Using conjugate gradient for inversion.
                                    - "GMRES" : Using GMRES for inversion.
                                    Defaults to 'exact'.
            max_inversions  :   int
                                The number of iterations for the iterative inversion algorithm.
                                Defaults to 30.
            relative_tol    :   float
                                The relative tolerance for the iterative inversion algorithm (CG or GMRES).
                                Defaults to 1e- 5.
            absolute_tol    :   float
                                The absolute tolerance for the iterative inversion algorithm (CG or GMRES).
                                Defaults to 1e-10.
            show_inversion_timings  :   bool
                                        Indicator to print time stamps at different steps of preconditioned inversion of the Hessian.    
            show_armijo_angle   :   bool
                                    Indicator to print the angle between the direction vector and the gradient.
            show_condition_number   :   bool
                                        Indicator to print the condition number of the Hessian matrix.
            observe_underflows  :   bool
                                    Indicator to observe the underflows


            Returns:
            --------
            Returns a dictionary where the keys are strings and the values are ndarrays or list.
            The following are the keys of the dictionary and the descriptions of their values:

            potential_f     :   ndarray, shape (n,)
                                The optimal Kantorovich potential f.
            potential_g     :   ndarray, shape (m,)
                                The optimal Kantorovich potential g.
            errors  :   list
                        The list of errors observed when checking conservation of mass.
            objective_values    :   list
                                    The list of objective values observed after each ascent update.
            linesearch_steps    :   list
                                    The list of ascent step sizes observed after each ascent update.
            timings_each_step   :   list
                                    The list of timings of each iteration of the algorithm.
        """
        i = 1
        while True: 
            print( "|"+"-" * 75 )
            print( "At iteration: ", i )
            start = time.time()
            # Computing the maximum exponent:
            max_f = np.max( self.f[:, None] - self.C, axis = 0 )# Shape: (m,)
            # Computing exponents
            exponents = ( self.f[:,None] - self.C - max_f[None, :] )/self.epsilon# Shape: (n,m)
            # Computing the exponentials:
            exp = self.a[:,None] * np.exp( exponents )# Shape: (n,m)
            # Computing the sum of exponentials:
            sum_exp = np.sum( exp, axis = 0 )# Shape: (m,)      
            # Computing the exponentials with normalization by the sum of the exponentials along the corresponding column:
            Gamma = exp/sum_exp[None,:]# Shape: (n,m)
            # Computing the gradient w.r.t f:
            self.gradient_f = self.a - np.sum( Gamma * self.b[None,:], axis = 1 )# Shape: (n,)
            # Computing the unnormalized Hessian:
            RowSum = np.sum( Gamma * self.b[None,:], axis = 1 )# Shape: (n,)
            self.Hessian = np.diag( RowSum ) - np.dot( Gamma, np.diag( self.b ) @ Gamma.T )# Shape: (n,n)
            if observe_underflows:
                underflow_threshold = np.log( np.finfo(np.float64).tiny )
                underflow_count = np.sum( exponents.flatten() < underflow_threshold)
                print( "|- Observing underflows and numerical inaccuracies arising from it:" )
                print( "|- |- System threshold for underflow: ", np.finfo(np.float64).tiny, "( in log: ", np.log( np.finfo(np.float64).tiny )," )" )
                print( "|- |- Range of values of the exponents: [", min( exponents.flatten() ),", ", max( exponents.flatten() ) ,"]| count of values violeting threshold ( < ", underflow_threshold,"): ", underflow_count )
                print( "|- |- Range of values of the exponentials: [", min( exp.flatten() ),", ", max( exp.flatten() ) ,"]| Proportion of values vanishing: ", underflow_count/( exponents.shape[0] * exponents.shape[1] )  ) 
                print( "|- |- Range of values of the gamma matri: [", min( Gamma.flatten() ),", ", max( Gamma.flatten() ) ,"]| Proportion of values vanishing: ", underflow_count/( exponents.shape[0] * exponents.shape[1] )  ) 
            # Compute solution of Ax = b:
            p_k, inversion_timings = self._preconditioned_inversion(    inversion_method = inversion_method, 
                                                                        max_inversions = max_inversions,
                                                                        rtol = relative_tol,
                                                                        atol = absolute_tol,
                                                                        show_inversion_timings = show_inversion_timings,
                                                                        show_condition_number = show_condition_number
                                                                    )
            # Record timings:
            self.inversion_timings.append( inversion_timings )
            print( "|- Time taken for the inversion: ", np.round( inversion_timings[-1], 5 ), " ms" )
            # Computing the update step size with using the Armijo condition:
            slope = np.dot( p_k, self.gradient_f )
            if show_armijo_angle:
                print( "|- cos( gradient, direction vector ): ", slope/( np.linalg.norm( p_k ) * np.linalg.norm( self.gradient_f ) ) ) 
            ### Inital step size:
            tau = 1
            ### Updated step size:
            backtracking_time_start = time.time()
            tau, reduction_count = self._backtracking( tau, p_k, slope )
            backtracking_time_end = time.time()
            backtracking_time = 1e3 * ( backtracking_time_end - backtracking_time_start )
            self.backtracking_info.append( {    'time' :   backtracking_time,
                                                'reduction_count'   :   reduction_count 
                                            }
                                        )
            print( "|- Time spent in backtracking: ", np.round( backtracking_time, 5), "ms| Reduction count: ", reduction_count )
            self.tau_list.append( tau )
            # Update f and g using damped Newton:
            self.f = self.f + tau * p_k# Shape: (n,)
            self.g = self._logexp_g( self.f )# Shape: (m,)
            end = time.time()
            # Record timings:
            time_dampedNewton = 1e3 * ( end - start )
            self.timings_each_step.append( time_dampedNewton )
            print( "|- Time taken to perform damped Newton: ",  np.round( time_dampedNewton, 5 ) ," ms." )
            print( "|"+"-" * 75 )
            # Computing the coupling matrix:
            P = self.a[:,None] * np.exp( ( self.f[:,None] + self.g[None,:]  - self.C )/self.epsilon ) * self.b[None,:]# Shape: (n,m), line (*)
            # Check conservation of mass: ||P 1_m - a||_1 + ||P^T 1_n - b||_1:
            self.errors.append( np.linalg.norm( np.sum( P, axis = 1 ) - self.a, ord = 1 )
                                +
                                np.linalg.norm( np.sum( P, axis = 0 ) - self.b, ord = 1 )
                              )
            # Evaluating objective function after the ascent update:
            value = self._objectivefunction( self.f, self.g )
            self.objective_values.append( value ) 
            # Condition to terminate the algorithm based on the estimation error of the marginals from the coupling:
            condition = self.errors[-1] > tol
            # Combining the two conditions:
            if i + 1 <= max_iterations and condition:
                i = i + 1
            else:
                print( "Terminating after iteration: ",i,"." )
                break
        # end while           
        # Change of convention because of line (*)
        self.f = self.f + self.epsilon * np.log( self.a )
        self.g = self.g + self.epsilon * np.log( self.b )
        return {
            "potential_f"       : self.f,
            "potential_g"       : self.g,
            "coupling_matrix"   : P,
            "errors"            : self.errors,
            "objective_values"  : self.objective_values,
            "linesearch_steps"  : self.tau_list,
            "inversion_timings" : self.inversion_timings,
            'backtracking_inf'  : self.backtracking_info,
            'timings_each_step' : self.timings_each_step
        }


## Code 4: Hybrid optimization(Damped Newton/Sinkhorn)

The following class in the semi-dual framework perform update of the potentials by comparing the updates of log-domain Sinkhorn and damped Newton with preconditioning, based on the increment in the objective function value.


In [ ]:
class hybrid_optimization:
    def __init__( self, C, a, b, f, epsilon, rho, c, null_vector, precond_vectors ):
        """

            Parameters:
            ----------- 
            C   :   ndarray, shape (n,m) 
                    It is the cost matrix between the points sampled from the point clouds.
            a   :   ndarray, shape (n,)
                    The probability histogram of the sample of size n.
            b   :   ndarray, shape (m,)
                    The probability histogram of the sample of size m.
            f   :   ndarray, shape (n,) 
                    The initial Kantorovich potential f.
            rho     :   float
                        Damping factor for the line search ascent step size.
            c   :   float
                    Sufficient increase parameter in the Armijo condition.
            epsilon     :   float
                            The regularization parameter.
            null_vector     :   ndarray, shape (n,)
                                The null vector of the Hessian to be used for null vector preconditioning.
            precond_vectors     :   list of ndarrays, shape (n,)
                                    The stack of preconditioning vectors obtained from the Hessian at the optimum obtained from the algorithm without any preconditioning,
                                    that is, semi-dual damped Newton with only null vector preconditioning and exact inversion.
        """
        self.C = C
        self.a = a
        self.b = b
        self.f = f
        self.epsilon = epsilon
        self.rho = rho           
        self.c = c
        self.null_vector = null_vector
        self.precond_vectors = precond_vectors
        self.tau_list = []
        self.errors = []
        self.objective_values = [] 
        self.update_indicator = {}
        self.backtracking_info = []
        self.timings_log_domainSinkhorn_each_step = []
        self.timings_dampedNewton_each_step = []
        
    def _objectivefunction( self, f, g ) :
        """ 

            Parameters:
            -----------
            f   :   ndarray, shape (n,)
                    The input Kantorovich potential f.
            g   :   ndarray, shape (m,)
                    The input Kantorovich potential g.
                    
            Returns: 
            --------
            Q_semi(f)   :   float
                            The value of semi-dual objective function obtained by evaluating Q_semi(f) = < f, a > + < g( f, C, a, epsilon ), b >,
                            where g( f, C, a, epsilon ) denotes the value of Kantorovich potential g evaluated using the Schrodinger-bridge equations between f and g.
         """
        Q_semi = np.dot( f, self.a ) + np.dot( g, self.b )
        return Q_semi
    
    def _f( self, g ):
        """

            Here we compute the value of the potential f by using its Schrodinger-bridge relation with the potential g: 
            f = - epsilon * log( ( b * exp( ( g - C )/epsilon ) )1_{m} ).

            Parameters:
            -----------
            g   :   ndarray, shape (m,)
                    The input Kantorovich potential g.
            Returns:
            --------
            ndarray, shape (n,)
            The value of potential f.
        """
        f = - self.epsilon * np.log( np.sum( self.b[None,:] * np.exp( ( g[None,:] - self.C ) /self.epsilon ), axis = 1 ) )
        return f# Shape: (n,)
    
    def _logexp_f( self, g ):
        """

            Here we incorporate the log-exp regularization method to the computation of the potential f by using its Schrodinger-bridge relation with the potential g: 
            f = - epsilon * log( ( b * exp( ( g - C - max_g )/epsilon ) )1_{m} ) - max_g,
            where max_g is the maximum value along each row of g - C.

            Parameters:
            -----------
            g   :   ndarray, shape (m,)
                    The input Kantorovich potential g.

            Returns:
            --------
            ndarray, shape (n,)
            The log-exp regularized value of potential f.
        """
        max_g = np.max(  g[None,:] - self.C, axis = 1 )
        f = - self.epsilon * np.log( np.sum( self.b[None,:] * np.exp( ( g[None,:] - self.C - max_g[:,None] ) /self.epsilon ), axis = 1 ) ) - max_g
        return f# Shape: (n,)

    def _g( self, f ):
        """

            Here we compute the value of the potential g by using its Schrodinger-bridge relation with the potential f: 
            g = - epsilon * log( 1_{n}^{T} ( a * exp( ( f - C )/epsilon ) ) ).

            Parameters:
            -----------
            f   :   ndarray, shape (n,)
                    The input Kantorovich potential f.
            Returns:
            --------
            ndarray, shape (m,)
            The value of potential g.
        """
        g = - self.epsilon * np.log( np.sum( self.a[:,None] * np.exp( ( f[:,None] - self.C )/self.epsilon ), axis = 0 ) )
        return g# Shape: (m,)
    
    def _logexp_g( self, f ):
        """

            Here we incorporate the log-exp regularization method to the computation of the potential g by using its Schrodinger-bridge relation with the potential f: 
            g = - epsilon * log( 1_{n}^{T} ( a * exp( ( f - C - max_f )/epsilon ) ) ) - max_f,
            where max_f is the maximum along each column of f - C.
            
            Parameters:
            -----------
            f   :   ndarray, shape (n,)
                    The input Kantorovich potential f.

            Returns:
            --------
            ndarray, shape (m,)
            The log-exp regularized value of potential g.

        """
        max_f = np.max( f[:,None] - self.C , axis = 0 )
        g =  - self.epsilon * np.log( np.sum( self.a[:,None] * np.exp( ( f[:,None] - self.C - max_f[None,:] )/self.epsilon ), axis = 0 ) )  - max_f
        return g# Shape: (m,)
            
    def _backtracking( self, tau, p, slope ):
        """

            Here we use the Armijo condition to decide the ascent step length for updating the potentials towards the ascent direction. 
            Parameters:
            -----------
            tau     :   float  
                        The ascent step size.
            p   :   ndarray, shape (n,)
                    The ascent direction.
            slope   :   float
                        It is the inner product of the gradient and p.
            Returns:
            --------
            tau     :   float
                        The updated ascent step size.
        """
        f_current = self.f# Shape: (n,)
        g_current = self._logexp_g( self.f )# Shape: (m,)
        reduction_count = 0     
        while True:
            f_updated = self.f + tau * p# Shape: (n,)
            g_updated = self._logexp_g( f_updated )# Shape: (m,)
            # Armijo condition
            condition = self._objectivefunction( f_updated, g_updated ) < self._objectivefunction( f_current, g_current ) + self.c * tau * slope
            if condition or np.isnan( self._objectivefunction( f_updated, g_updated ) ):
                tau = self.rho * tau  
                reduction_count += 1
            else:
                break
        # end while
        return tau, reduction_count
    
    def _preconditioned_inversion( self, rtol, atol, inversion_method, max_inversions, show_inversion_timings, show_condition_number ):    
        """

            Parameters:
            -----------
            inversion_method    :   str
                                    The method of inversion to be used for the Hessian. The following are the options:
                                    - "exact" : Exact inversion.
                                    - "cg" : Using conjugate gradient for inversion.
                                    - "GMRES" : Using GMRES for inversion.
            max_inversions  :   int
                                The number of iterative inversions to be performed to obtain the inverse of the Hessian followed by obtaining the ascent direction.
            rtol    :   float
                        The value of relative tolerance which is a hyperparameter to the iterative inversion algorithm, here it is conjugate gradient (CG) or GMRES.
            atol    :   float
                        The value of absolute tolerance which is a hyperparameter to the iterative inversion algorithm, here it is conjugate gradient (CG) or GMRES.    
            show_inversion_timings  :   bool
                                        Indicator to print time stamps at different steps of preconditioned inversion of the Hessian.       
            show_condition_number   :   bool
                                        Indicator to print the condition number of the Hessian matrix.


            Returns:
            --------
            Returns a tuple containing the optimal ascent direction vector p and the recorded timings of various steps of the algorithm. 
            The following are their descriptions:

            p   :   ndarray, shape: (n,)
                    The optimal ascent direction vector.
            timings     :   list
                            The list of timestamps recorded.
        """

        def printing_time( statements ):
            """
                Function to enable the option to print time stamps at different steps.
                Parameters:
                -----------
                statements  :   List of strings to be printed in a line.
            """
            if show_inversion_timings:
                print( " ".join( statements ) )
                
        timings = []
        if show_inversion_timings or show_condition_number or inversion_method in [ 'cg', 'GMRES' ]:
            print( "|"+"-" * 75 )
        start = time.time()
        # Record list of unwinding transformations on final result:
        unwinding_transformations = []
        # Construct modified Hessian:  
        diag = 1/np.sqrt( self.a )          
        self.modified_Hessian = diag[:,None] * self.Hessian * diag[None,:]
        # Dummy variable to work on:
        matrix = self.modified_Hessian
        # Preconditioning along null vector:
        vector = self.null_vector# Shape: (n,)
        vector = vector/np.linalg.norm( vector )
        vector_E = vector
        # Transforming the gradient by conjugation:
        gradient = diag[:,None] * self.gradient_f[:,None]
        # Unwinding transformation to obtain the original direction vector i.e., without any conjugation:
        unwinding_transformations.append( lambda x : diag[:,None] * x )
        end = time.time()
        # Record timings:
        interval = 1e3 * ( end - start )
        timings.append( interval )
        printing_time( [ "|--|- Time required for initial preconditioning: ", str( np.round( interval, 5 ) ), "ms" ])
        # Conditioning with other vectors
        #  Naming conventions:
        #  y = Preconditioning vectors as a numpy matrix n by k
        #  matrix = our matrix A to precondition
        #  We only form the data y and z such that
        #  P = id + z*y.T
        start0 = time.time()
        y = np.array( self.precond_vectors).T # Matrix of size n by k
        # Compute eigenvalues:
        Ay = np.dot( matrix, y )
        eigenvalues = np.sum( y * Ay, axis = 0 )
        # Compute data for P = id + y*diag(values)*y.T:
        values = ( 1/np.sqrt(eigenvalues) - 1 )# Vector of size k
        z = y * values[None,:]
        end = time.time()
        # Record timings:
        interval = 1e3 * ( end - start0 )
        timings.append( interval )
        printing_time( [ "|--|- Time required for preconditioning matrix formation: ", str( np.round( interval, 5 ) ), "ms" ] )
        # Changing A=matrix to PAP:
        start2 = time.time()
        def _apply_P( vector ):
            """

                Function mapping v to Pv, where P = Id + z*y.T.

                Parameters:
                -----------
                vector  :   ndarray, shape: (n,)

                Returns:
                --------
                ndarray, shape: (n,)  
                vector obtained from the map Pv.
            """
            return  vector + z @ ( y.T @ vector ) 
        def _preconditioned_map( vector ):
            """

                Function mapping v to P(A+E)Pv,
                    where   P = P = Id + z*y.T
                            A   :   ndarray, shape: (n,n)                                            
                            E   :   ndarray, shape: (n,n)
                                    outer product of the null vector with itself.       
                Parameters:
                -----------
                    vector  :   ndarray, shape: (n,)
                
                Returns:
                --------
                ndarray, shape: (n,)
                The preconditioned input vector. 
            """
            vector = _apply_P( vector ) 
            vector = np.dot( matrix, vector )  + vector_E * np.dot( vector_E, vector )
            vector = _apply_P( vector ) 
            return vector
        # Apply P
        # At beginning on gradient
        # At the end 
        # Preconditioning the gradient:
        gradient = _apply_P( gradient )
        # The transformation to precondition the direction vector:
        unwinding_transformations.append( lambda x : _apply_P(x) )
        end = time.time()
        # Record timings:
        interval = 1e3 * ( end - start2 )
        timings.append( interval )
        printing_time( [ "|--|- Time required for changing A to PAP: ", str( np.round( interval, 5 ) ), "ms" ] )
        #
        # Solve either iteratively using CG or exactly:
        start3 = time.time()
        if inversion_method != "exact":
            ## Inversion counter:
            inversion_count = [0]
            ## Function to increment counter:
            callback = lambda x: inversion_count.__setitem__(0, inversion_count[0] + 1)
            ## Print condition number:
            if show_condition_number:
                eig, v = np.linalg.eigh( matrix )
                sorting_indices = np.argsort( eig )
                eig = eig[ sorting_indices ]
                v = v[ :, sorting_indices ]
                print( "|--|- Observing eigenvalues and condition number:" )
                print( "|--|- |- List of smallest 10 eigenvalues: [", ", ".join( list( map( str, eig[ : 10 ] ) ) ), "]" )
                print( "|--|- |- List of largest  10 eigenvalues: [", ", ".join( list( map( str, eig[ - 10 : ]  ) ) ), "]" ) 
                print( "|--|- |- Conition number of the Hessian: ", np.linalg.cond( matrix ) )
            self.m  = matrix
            A = scipy.sparse.linalg.LinearOperator( ( self.m.shape[0], self.m.shape[1] ), matvec = _preconditioned_map ) 
            if inversion_method == 'cg':
                print( "|--|- Inverting using conjugate gradient:" )
                inverse, exit_code = scipy.sparse.linalg.cg(    A,
                                                                gradient, 
                                                                x0 = gradient, 
                                                                maxiter = max_inversions, 
                                                                rtol = rtol, 
                                                                atol = atol,
                                                                callback = callback 
                                                            )
                    # print( "  --- CG exit code: ", exit_code)
            else:
                print( "|--|- Inverting using GMRES:" )
                inverse, exit_code = scipy.sparse.linalg.gmres(     A,
                                                                    gradient, 
                                                                    x0 = gradient, 
                                                                    maxiter = max_inversions, 
                                                                    rtol = rtol, 
                                                                    atol = atol,
                                                                    callback = callback 
                                                                )
                # print( "  --- GMRES exit code: ", exit_code)
            p_k = self.epsilon * inverse
            p_k = p_k.reshape( ( p_k.shape[0], 1 ) )# For some reason, this outputs (n,) and the next line outputs (n,1)
            if exit_code == 0:
                inversion_convergence = 'yes'
            else:
                inversion_convergence = 'no'
            print( "|--|- |- Number of iterative inversions: ", inversion_count[0] )
            print( "|--|- |- Convergence status: ", inversion_convergence )
        else:
            # Preconditioning along null vector:                                     
            vector = vector.reshape( ( len(vector), 1 ) )
            matrix = matrix + np.dot( vector, vector.T )     
            # True Preconditioning for exact inverse: 
            B = np.dot( Ay, z.T )
            C = z @ np.dot( y.T, Ay ) @ z.T
            matrix = matrix + B + B.T + C
            self.Hessian_stabilized = - matrix/self.epsilon
             # Print condition number:
            if show_condition_number:
                eig, v = np.linalg.eigh( matrix )
                sorting_indices = np.argsort( eig )
                eig = eig[ sorting_indices ]
                v = v[ :, sorting_indices ]
                print( "|--|- Observing eigenvalues and condition number:" )
                print( "|--|- |- List of smallest 10 eigenvalues: [", ", ".join( list( map( str, eig[ : 10 ] ) ) ), "]" )
                print( "|--|- |- List of largest  10 eigenvalues: [", ", ".join( list( map( str, eig[ - 10 : ]  ) ) ), "]" ) 
                print( "|--|- |- Conition number of the Hessian: ", np.linalg.cond( matrix ) )
            # Performing exact inversion:
            p_k = - np.linalg.solve( self.Hessian_stabilized, gradient )
        end = time.time()
        # Record timings:
        interval = 1e3 * ( end - start3 )
        timings.append( interval )
        printing_time( [ "|--|- Time taken to invert the linear system for p_k: ", str( np.round( interval, 5 )), "ms" ] )
        start4 = time.time()
        # Unwind:
        for transform in unwinding_transformations:
          p_k = transform( p_k )
        end = time.time()
        # Record timings:
        interval = 1e3 * ( end - start4 )
        timings.append( interval )
        printing_time( [ "|--|- Time taken for unwinding: ", str( np.round( interval, 5 ) ), "ms" ] )
        interval = 1e3 * ( end - start )
        timings.append( interval )
        printing_time( [ "|--|- Time taken for the complete code block: ", str( np.round( interval, 5 ) ), "ms"] )
        if show_inversion_timings or show_condition_number or inversion_method in [ 'cg', 'GMRES' ]:
            print( "|"+"-" * 75 )
        return p_k.flatten(), timings
        
    def _optimize( self, tol_obj = 1e-12, tol_err = 1e-12, max_iterations = 50, inversion_method = "exact", max_inversions = 30, relative_tol = 1e-5, absolute_tol = 1e-10, show_inversion_timings = False, show_armijo_angle = False, show_condition_number = False, observe_underflows = False ):
        """
        
            Parameters:
            -----------
            tol_obj     :   float
                            Tolerance to terminate the algorithm based on the comparison of the objective values using the 
                            updates of the potentials from log-domain Sinkhorn and damped Newton. 
                            Defaults to 1e-12.
            tol_err :       float
                            Tolerance to terminate the algorithm based on the error in estimating the marginals of the coupling.
                            Defaults to 1e-12.
            max_iterations  :   int
                                The maximum number of iterations for the optimization algorithm.
                                Defaults to 50.
            inversion_method    :   str
                                    The method of inversion to be used for the Hessian. The following are the options:
                                    - "exact" : Exact inversion.
                                    - "cg" : Using conjugate gradient for inversion.
                                    - "GMRES" : Using GMRES for inversion.
                                    Defaults to 'exact'.
            max_inversions  :   int
                                The number of iterations for the iterative inversion algorithm.
                                Defaults to 30.
            relative_tol    :   float
                                The relative tolerance for the iterative inversion algorithm (CG or GMRES).
                                Defaults to 1e- 5.
            absolute_tol    :   float
                                The absolute tolerance for the iterative inversion algorithm (CG or GMRES).
                                Defaults to 1e-10.
            show_inversion_timings  :   bool
                                        Indicator to print time stamps at different steps of preconditioned inversion of the Hessian.    
            show_armijo_angle   :   bool
                                    Indicator to print the angle between the direction vector and the gradient.
            show_condition_number   :   bool
                                        Indicator to print the condition number of the Hessian matrix.
            observe_underflows  :   bool
                                    Indicator to observe the underflows
                                    
            Returns:
            --------
            Returns a dictionary where the keys are strings and the values are ndarrays or list.
            The following are the keys of the dictionary and the descriptions of their values:

            potential_f     :   ndarray, shape (n,)
                                The optimal Kantorovich potential f.
            potential_g     :   ndarray, shape (m,)
                                The optimal Kantorovich potential g.
            errors  :   list
                        The list of errors observed when checking conservation of mass.
            objective_values    :   list
                                    The list of objective values observed after each ascent update.
            linesearch_steps    :   list
                                    The list of ascent step sizes observed after each ascent update.
            update_indicator    :   list
                                    A list of indicator strings where each entry is  the string 
                                    "log-domain Sinkhorn" if the potentials at this iteration were update using the log-domain Sinkhorn
                                    and "damped Newton" otherwise.
            backtracking_info   :   list
                                    The list of dictionaries where each dictionary contains the time taken for backtracking and the reduction count
                                    for each iteration. 
            timings_each_step   :   list 
                                    The list of lists of the timings recorded for log-domain Sinkhorn and damped Newton .    
        """
        i = 1
        while True: 
            print( "|"+"-" * 75 )
            print( "At iteration: ", i )
            #---------------------------
            # Log-domain Sinkhorn update
            #---------------------------
            print( "|- Log-domain Sinkhorn update" )
            start = time.time()
            ## Update f and g using log-domain Sinkhorn:
            g_log_domainSinkhorn = self._logexp_g( self.f )# Shape: (m,)
            f_log_domainSinkhorn = self._logexp_f( g_log_domainSinkhorn )# Shape: (n,)
            end = time.time()
            time_log_domainSinkhorn = 1e3 * ( end - start )
            ## Record timings:
            self.timings_log_domainSinkhorn_each_step.append( time_log_domainSinkhorn )
            print( "|-- Time taken to perform log-domain Sinkhorn: ",  np.round( time_log_domainSinkhorn, 5 ) ," ms" )
            ## Computing the objective value with the potentials updated using log-domain Sinkhorn:
            log_domainSinkhorn_objective_value = self._objectivefunction( f_log_domainSinkhorn, g_log_domainSinkhorn )
            #---------------------
            # Damped Newton update
            #---------------------
            print( "|- Damped Newton update" )
            start = time.time()
            ## Computing the maximum exponent:
            max_f = np.max( self.f[:, None] - self.C, axis = 0 )# Shape: (m,)
            ## Computing exponents
            exponents = ( self.f[:,None] - self.C - max_f[None, :] )/self.epsilon# Shape: (n,m)
            ## Computing the exponentials:
            exp = self.a[:,None] * np.exp( exponents )# Shape: (n,m)
            ## Computing the sum of exponentials:
            sum_exp = np.sum( exp, axis = 0 )# Shape: (m,)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  
            ## Computing the exponentials with normalization by the sum of the exponentials along the corresponding column:
            Gamma = exp/sum_exp[None,:]# Shape: (n,m)
            ## Computing the gradient w.r.t f:
            self.gradient_f = self.a - np.sum( Gamma * self.b[None,:], axis = 1 )# Shape: (n,)
            ## Computing the unnormalized Hessian:
            RowSum = np.sum( Gamma * self.b[None,:], axis = 1 )# Shape: (n,)
            self.Hessian = np.diag( RowSum ) - np.dot( Gamma, np.diag( self.b ) @ Gamma.T )# Shape: (n,n)
            if observe_underflows:
                underflow_threshold = np.log( np.finfo(np.float64).tiny )
                underflow_count = np.sum( exponents.flatten() < underflow_threshold)
                print( "|-- Observing underflows and numerical inaccuracies arising from it:" )
                print( "|-- |- System threshold for underflow: ", np.finfo(np.float64).tiny, "( in log: ", np.log( np.finfo(np.float64).tiny )," )" )
                print( "|-- |- Range of values of the exponents: [", min( exponents.flatten() ),", ", max( exponents.flatten() ) ,"]| count of values violeting threshold ( < ", underflow_threshold,"): ", underflow_count )
                print( "|-- |- Range of values of the exponentials: [", min( exp.flatten() ),", ", max( exp.flatten() ) ,"]| Proportion of values vanishing: ", underflow_count/( exponents.shape[0] * exponents.shape[1] )  ) 
                print( "|-- |- Range of values of the gamma matrix: [", min( Gamma.flatten() ),", ", max( Gamma.flatten() ) ,"]| Proportion of values vanishing: ", underflow_count/( exponents.shape[0] * exponents.shape[1] )  ) 
            ## Compute solution of Ax = b:
            p_k, inversion_timings  =   self._preconditioned_inversion(     inversion_method = inversion_method, 
                                                                            max_inversions = max_inversions,
                                                                            rtol = relative_tol,
                                                                            atol = absolute_tol,
                                                                            show_inversion_timings = show_inversion_timings,
                                                                            show_condition_number = show_condition_number
                                                                        )
            print( "|-- Time taken to compute the direction vector: ", np.round( inversion_timings[-1], 5 ), "ms" )
            ## Computing the update step size with using the Armijo condition:  
            slope = np.dot( p_k, self.gradient_f )
            if show_armijo_angle:
                print( "|-- cos( gradient, direction vector ): ", slope/( np.linalg.norm( p_k ) * np.linalg.norm( self.gradient_f ) ) ) 
            ### Inital step size:
            tau = 1
            ### Updated step size:
            backtracking_time_start = time.time()
            tau, reduction_count = self._backtracking( tau, p_k, slope )
            backtracking_time_end = time.time()
            backtracking_time = 1e3 * ( backtracking_time_end - backtracking_time_start )
            self.backtracking_info.append( {    'time' :   backtracking_time,
                                                'reduction_count'   :   reduction_count 
                                            }
                                        )
            print( "|-- Time spent in backtracking: ", np.round( backtracking_time, 5), "ms| Reduction count: ", reduction_count )
            self.tau_list.append( tau )
            ## Update f and g using damped Newton:
            f_damped_Newton = self.f + tau * p_k# Shape: (n,)
            g_damped_Newton = self._logexp_g( f_damped_Newton )# Shape: (m,)
            end = time.time()
            time_dampedNewton = 1e3 * ( end - start )
            ## Record timings:
            self.timings_dampedNewton_each_step.append( time_dampedNewton )
            print( "|-- Time taken to perform damped Newton: ",  np.round( time_dampedNewton, 5 ) ," ms" )
            ## Computing the objective value with the potentials updated using damped Newton:
            damped_Newton_objective_value = self._objectivefunction( f_damped_Newton, g_damped_Newton )
            # -----------------
            # Comparing updates
            # -----------------
            ## Compare the objective function value for the potentials from the two methods and choosing the best update:
            if log_domainSinkhorn_objective_value > damped_Newton_objective_value:
                print( "|- Updating using log-domain Sinkhorn" )
                self.f = f_log_domainSinkhorn# Shape: (n,)
                self.g = g_log_domainSinkhorn# Shape: (m,)
                self.update_indicator[ i - 1 ] = "log-domain Sinkhorn" 
            else:
                print( "|- Updating using damped Newton" )
                self.f = f_damped_Newton# Shape: (n,)
                self.g = g_damped_Newton# Shape: (m,)
                self.update_indicator[ i - 1 ] = "Damped Newton"
            print( "|"+"-" * 75 )
            # Computing the coupling matrix:
            P = self.a[:,None] * np.exp( ( self.f[:,None] + self.g[None,:]  - self.C )/self.epsilon ) * self.b[None,:]# Shape: (n,m), line (*)
            # Check conservation of mass: ||P 1_m - a||_1 + ||P^T 1_n - b||_1:
            self.errors.append( np.linalg.norm( np.sum( P, axis = 1 ) - self.a, ord = 1 )
                                +
                                np.linalg.norm( np.sum( P, axis = 0 ) - self.b, ord = 1 )
                              )
            # Evaluating objective function after the ascent update:
            value = self._objectivefunction( self.f, self.g )
            self.objective_values.append( value ) 
            # Condition to terminate the algorithm based on the objective values using the potentials updated using the log-domain Sinkhorn and damped Newton:
            condition_obj = abs( log_domainSinkhorn_objective_value - damped_Newton_objective_value ) > tol_obj
            # Condition to terminate the algorithm based on the estimation error of the marginals from the coupling:
            condition_err = self.errors[-1] > tol_err
            # Combining the two conditions:
            condition = condition_obj or condition_err
            if i + 1 <= max_iterations and condition:
                i = i + 1
            else:
                print( "Terminating after iteration: ", i  )
                break
        # end while           
        # Change of convention because of line (*)
        self.f = self.f + self.epsilon * np.log( self.a )
        self.g = self.g + self.epsilon * np.log( self.b )
        return {
            "potential_f"       : self.f,
            "potential_g"       : self.g,
            'Coupling_matrix'   : P,
            "errors"            : self.errors,
            "objective_values"  : self.objective_values,
            "linesearch_steps"  : self.tau_list,
            "update_indicator"  : self.update_indicator,
            'backtracking_info' : self.backtracking_info,
            "timings_each_step" : { 'timings_log_domainSinkhorn'    :    self.timings_log_domainSinkhorn_each_step, 
                                    'timings_dampedNewton'   :   self.timings_dampedNewton_each_step 
                                   }
        }


# Experiments

In [ ]:
experiment_results = {
    "results_Sinkhorn"  :   {},
    "results_log_domain_Sinkhorn"    :   {},
    "results_damped_Newton_exact_inversion"  :   {},
    "results damped Newton_iterative_inversion"  :   {},
    "results_hybrid_optimization_exact_inversion"  :   {},
    "results_hybrid_optimization_iterative_inversion"  :   {}
}

In [ ]:
epsilons = [ 1.0,  0.5, 0.1, 0.05, 0.01, 0.005, 0.001, 0.0009, 0.0007, 0.0005 ]
# epsilons = [ 1.0, 0.5, 0.1, 0.05, 0.01 ]

#### Helper functions

In [ ]:
"""To compute distance matrix"""
def distmat( x, y ):
    return np.sum( x ** 2, 0 )[:,None] + np.sum( y ** 2, 0 )[None,:] - 2 * x.transpose().dot( y )
"""To Normalise a vector"""
normalize = lambda a: a/np.sum( a )
"""To Compute P"""
def GetP( u, K, v ):
    return u[:,None] * K * v[None,:]

In [ ]:
def generate_data( N ):
    """
     N is a list of the sizes for sampling the data from x and y.
    """
    # Sampling from a square
    x = np.random.rand( 2, N[0] ) - 0.5
    # Sampling from an annulus
    theta = 2 * np.pi * np.random.rand( 1, N[1] )  
    r = 0.8 + 0.2 * np.random.rand( 1, N[1] )
    y = np.vstack( ( r * np.cos( theta ), r * np.sin( theta ) ) )
    return x, y

#### Sampling point clouds

In [ ]:
# Here N[0] is the number of rows and columns of the Hessian in the semi-dual framework
N = [ 600, 500 ] 
x, y = generate_data( N )                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   
# Cost matrix
C = distmat( x, y )
# a and b
a = normalize( np.ones( N[0] ) )
b = normalize( np.ones( N[1] ) )

## I. Sinkhorn: Highlighting the failure of the algorithm for small $\varepsilon$

Here we can observe that for $\varepsilon \leq 0.0009$ the algorithm terminates because of overflow and underflow of values.

In [ ]:
# Sinkhorn
print( "Sinkhorn... " )
print( "Doing for (",N[0], N[1],")." )
for epsilon in epsilons:
  print( "For epsilon = "+str(epsilon)+":" )    
  # Kernel                                                                                                                                                                
  K = np.exp( - C/epsilon )
  print( " |- Iterating" )
  u = a
  v = b 
  optimizer = sinkhorn( K, a, b, u, v, epsilon )
  start = time.time()
  experiment_results[ "results_Sinkhorn" ][ epsilon ] = optimizer._optimize( max_iterations = int(1e100) )
  end = time.time()
  experiment_results[ "results_Sinkhorn" ][ epsilon ][ "total_time" ] = end - start
  print( " |- Computing P" )
  print( "" )
# end for


### Error plot

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )
plt.style.use( "seaborn-v0_8-notebook" )
plt.figure( figsize = ( 20, 7 ) )
plt.title( r"$E_{\alpha,\beta}\left(P_{\varepsilon}\right) = \|P_{\varepsilon}\mathbb{1}_{m} -\alpha\|_1+\|P_{\varepsilon}^{T}\mathbb{1}_{n} -\beta\|_1$" ) 
for epsilon in epsilons:
  errors = np.asarray( experiment_results[ "results_Sinkhorn" ][ epsilon ][ 'errors' ] )
  plt.plot( errors, label = r'Sinkhorn for $\varepsilon = $'+ str(epsilon), linewidth = 2 )
# end for
plt.xlabel( " Number of iterations " )
plt.ylabel( r"$E_{\alpha,\beta}\left(P_{\varepsilon}\right)$" )
plt.yscale( 'log' )
plt.legend( loc = "upper right", fontsize = "large" )
plt.savefig( image_folder_path + "/Error_Sinkhorn.pdf", format = 'pdf' )
plt.show()

## II. Log-domain Sinkhorn: To get the preconditioning vectors

In [ ]:
# Log-domain Sinkhorn
print( "Log-domain Sinkhorn... " )
print( "Doing for (",N[0],N[1],")." )
a = normalize( np.ones( N[0] ) )
b = normalize( np.ones( N[1] ) )
# Cost matrix
C = distmat( x, y )
for epsilon in epsilons:
  print( "For epsilon = "+str(epsilon)+":" )    
  print( " |- Iterating" )
  optimizer = log_domainSinkhorn( a, b, C, epsilon )
  start = time.time()
  experiment_results[ "results_log_domain_Sinkhorn" ][ epsilon ] = optimizer._optimize( max_iterations = int(1e100) )
  end = time.time()
  experiment_results[ "results_log_domain_Sinkhorn" ][ epsilon ][ 'total_time' ] = end - start
  print( "" )
# end for

### Error plot

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )
plt.style.use( "seaborn-v0_8-notebook" )
plt.figure( figsize = ( 20, 7 ) )
plt.title( r"$E_{\alpha,\beta}\left(P_{\varepsilon}\right) = \|P_{\varepsilon}\mathbb{1}_{m} -\alpha\|_1+\|P_{\varepsilon}^{T}\mathbb{1}_{n} -\beta\|_1$" ) 
for epsilon in epsilons:
  errors = np.asarray( experiment_results[ "results_log_domain_Sinkhorn" ][ epsilon ][ 'errors' ] )
  plt.plot( errors, label = r'Log-domain Sinkhorn for $\varepsilon = $' + str(epsilon), linewidth = 2 )
# end for
plt.xlabel( " Number of iterations " )
plt.ylabel( r"$E_{\alpha,\beta}\left(P_{\varepsilon}\right)$" )
plt.yscale( 'log' )
plt.legend( loc = "upper right", fontsize = "large" )
plt.savefig( image_folder_path + "/Error_log_domain_Sinkhorn.pdf", format = 'pdf' )
plt.show()

### Computing the Hessians for different values of $\varepsilon$

In [ ]:
log_domainSinkhorn_Hessians = {}
for epsilon in epsilons:
    f = experiment_results[ "results_log_domain_Sinkhorn" ][ epsilon ][ "potential_f" ]
    # Computing the maximum exponent
    max_f = np.max( f[:, None] - C  , axis = 0 )# Shape: (m,)
    # Computing the exponentials
    exp = a[:,None] * np.exp( ( f[:,None] - C - max_f[None, :] )/epsilon )# Shape: (n,m)
    # Computing the sum of exponentials
    sum_exp = np.sum( exp, axis = 0 )# Shape: (m,)
    # Computing the exponentials with normalization by the sum of the exponentials along the corresponding column
    normalized_exp = exp/sum_exp# Shape: (n,m)
    # Computing the unnormalized Hessian
    RowSum = np.sum( normalized_exp * b[None,:], axis = 1 )# Shape: (n,)
    Hessian = np.diag( RowSum ) - np.dot( normalized_exp, np.diag( b ) @ normalized_exp.T )# Shape: (n,n)
    diag = 1/np.sqrt( a )
    log_domainSinkhorn_Hessians[ epsilon ] =  diag[:,None] * Hessian * diag[None,:]

### Spectral statistics

In [ ]:
def spectral_decomposition( mat ):
    eig, v = np.linalg.eigh( mat )
    sorting_indices = np.argsort( eig )
    eig = eig[ sorting_indices ]
    v = v[ :, sorting_indices ]
    print( "List of smallest eigenvalues: ", eig[ : 10 ] )
    print( "List of largest  eigenvalues: ", eig[ - 10 : ] )
    return eig, v

In [ ]:
eigs = []
eigvecs = []
for epsilon in epsilons:
    print( "Spectral statistics of Hessian for epsilon = "+str(epsilon) )
    result = log_domainSinkhorn_Hessians[ epsilon ] 
    ev = spectral_decomposition( result )
    eigs.append( ev[0] )    
    eigvecs.append( ev[1] )
    print( "" )  
# end for

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )
plt.style.use('default')
fig, ax = plt.subplots( figsize = ( 5, 30 ), nrows =  len( epsilons ), ncols = 1, sharey = True )
plt.title( " Histogram of eigenvalues. " )
for i in range( len( epsilons ) ):
    ax[i].hist( eigs[i], bins = 50 )
    ax[i].set_title( r"$\varepsilon$: " + str( epsilons[i] ) )
    ax[i].set_xlabel( " Eigenvalues " )
    ax[i].set_yscale( "log" )
    ax[i].set_xlim( 0, 1 )
# end for
plt.subplots_adjust( wspace = 0, hspace = 0.5 ) 
plt.tight_layout()
plt.savefig( image_folder_path + "/Spectral_plot.pdf", format = 'pdf' )
plt.show()

#### Building preconditioning vectors

##### Observing the performance of np.eigh in computing the eigenvalues and eigenvectors for matrices of varying sizes

In [ ]:
# !pip install perfplot

In [ ]:
# plt.rcParams.update( { 'font.size' : 12 } )
# plt.style.use( "seaborn-v0_8-notebook" )
# import perfplot
# perfplot.show(
#     setup = lambda n: np.random.rand( n, n ),
#     kernels = [ lambda A : np.linalg.eigh( A ) ],
#     n_range = [ 10 ** k for k in range( 1, 5 ) ],
#     xlabel = "Matrix dimension",
#     title = "Performance of np.linalg.eigh",
#     logx = True,
#     logy = True,
#     time_unit = "s"# Time unit in seconds
# )
# plt.close()

In [ ]:
def build_preconditioners( num_eigs, modified_Hessian, ansatz = True ):
    # Diagonalize
    start = time.time()
    eigenvalues, eigenvectors = np.linalg.eigh( modified_Hessian )
    end = time.time()
    diagnalize_time = ( end - start ) 
    sorting_indices = np.argsort( eigenvalues  )
    eigenvalues  = eigenvalues[ sorting_indices ]
    eigenvectors = eigenvectors[ :, sorting_indices ]
    # Form null vector
    if not ansatz:
        null_vector = eigenvectors[:, 0]
    else:
        null_vector = np.ones( N[0] ) 
        norm = np.sqrt( N[0] )
        null_vector = null_vector/norm
    # Form other vectors
    indices = []
    for i in range( num_eigs ):
        indices.append( i + 1 )
    # end for
    precond_vectors = []
    for index in indices:
        precond_vectors.append( eigenvectors[ :, index ] )
    # end for
    return null_vector, precond_vectors, diagnalize_time

### Effect of preconditioning on spectrum for various number of preconditioning eigenvectors

In [ ]:
num_eigs = [ 0, 10, 20, 30, 40, 50, 75, 100 ]
preconditioned_Hessians = {}
for numeigs  in  range( len( num_eigs ) ):
    preconditioned_Hessians[ num_eigs[ numeigs ] ] = []
    for i in  range( len( epsilons ) ):
        result = log_domainSinkhorn_Hessians[ epsilons[i] ]
        if num_eigs[ numeigs ] != 0:
            null_vector, precond_vectors, _ = build_preconditioners( num_eigs[ numeigs ], result, ansatz = False )
            w = np.array( precond_vectors ).T# Matrix of size n by k
            # Compute eigenvalues
            Aw = np.dot( result, w )
            eigenvalues = np.sum( w * Aw, axis = 0 )
            # Compute P_matrix = id + y*diag(values)*y.T
            values = ( 1/np.sqrt(eigenvalues) - 1 )# Vector of size k
            z = w * values[None,:]
            B = np.dot( Aw, z.T )
            D = z @ np.dot( w.T, Aw ) @ z.T
            result = result + B + B.T + D
        preconditioned_Hessians[ num_eigs[ numeigs ] ].append( result )
    # end for
# end for                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [ ]:
eigs = {}
for numeigs in  range( len( num_eigs ) ):
    eigs[ num_eigs[ numeigs ] ] = []                                                                                                                                                                                             
    for i in range( len( epsilons ) ):
        eps = epsilons[i]
        print( "Spectral statistics of Hessian for epsilon = " +str(eps) )
        ev = spectral_decomposition( preconditioned_Hessians[ num_eigs[ numeigs ] ][ i ] )
        eigs[ num_eigs[ numeigs ] ].append( ev[0] )
        print("")       
    # end for
# end for

#### Sprectral plots showing the change in the spectrum of Hessian after preconditioning with different number of preconditioning vectors

In [ ]:
plt.style.use('default')
plt.rcParams.update( { 'font.size' : 40 } )
fig, ax = plt.subplots( figsize = ( 120, 90 ), nrows = len(num_eigs), ncols = len(epsilons), sharey = True, sharex = False )
p = np.log10( 0.5 )   
for numeigs in range( len( num_eigs ) ):    
    for i in range( len( epsilons ) ):
        ax[ numeigs ][i].hist( eigs[ num_eigs[ numeigs ] ][i], 50, rwidth = 0.9 )
        ax[ numeigs ][i].set_title( " k = "+str(num_eigs[ numeigs ])+r", $\varepsilon$ = " +str(epsilons[i])+ "" )
        ax[ numeigs ][i].set_ylim( ymin = 10 ** p )
        ax[ numeigs ][i].set_yscale( "log" )    
    # end for
# end for
ax[ len(num_eigs) - 1 ][ len( epsilons ) - 1 ].set_xticks( [ 0, 1 ] )  
plt.subplots_adjust( wspace = 0.1, hspace = 0.2 )
plt.savefig( image_folder_path + "/Effect_of_preconditioning.pdf", format = 'pdf' )
plt.show()

## III. Optimization in the semi-dual framework using damped Newton with preconditioning and hybrid optimization method


### Preconditioning $\varepsilon = 0.5$

### i) Using exact method for Hessian inversion

#### Damped Newton with preconditioning

In [ ]:
# Number of preconditioning eigenvectors, adjusted to be within the bounds of the array dimensions
num_eigs = 35
# Choosing the epsilon corresponding to which the Hessian is used to obtain the preconditioning vectors
preconditioning_epsilon = 0.5
null_vector, precond_vectors, _ = build_preconditioners( num_eigs, log_domainSinkhorn_Hessians[ preconditioning_epsilon ], ansatz = False )

In [ ]:
print( " Doing for (",N[0], N[1],"). " )
# Damping factor for ascent step-size
rho = 0.4
# Sufficient increase parameter in the Armijo condition 
c = 0.1
# Number of iteratis
num_iterations = 50
# Inversion method
inversion_method = "exact"
# Indicator to plot time histograms
plot_time_histogram = False
f = None
for epsilon in epsilons :
    print( "For epsilon = "+str(epsilon)+":" )    
    # Initializing potential f
    if f is None:
        f = a * 0  
    print( " Iterating" )
    optimizer =  semi_dual_damped_Newton_with_preconditioning(  C,
                                                                a,
                                                                b,
                                                                f,
                                                                epsilon,
                                                                rho,
                                                                c,
                                                                null_vector,
                                                                precond_vectors[:]
                                                            )    
    start = time.time()                                                  
    experiment_results[ "results_damped_Newton_exact_inversion" ][ epsilon ] = optimizer._optimize(     max_iterations = num_iterations,
                                                                                                        inversion_method = inversion_method,
                                                                                                        show_inversion_timings = False,# To see timestamps at different steps during the inversion of the Hessian
                                                                                                        show_armijo_angle = False,# To see the angle between the obtained direction vector and the gradient
                                                                                                        show_condition_number = False,# To see the condition number of the matrix just before inversion
                                                                                                        observe_underflows = False# To see underflow appearing                                                                                                    )                                                                                                    
                                                                                                    )
    end = time.time() 
    # Recording time taken for each epsilon
    experiment_results[ "results_damped_Newton_exact_inversion" ][ epsilon ]['total_time'] = end - start
    print( "" )
    if plot_time_histogram:
        # Compute mean and standard deviation of the timings
        mean_damped_Newton_time, std_damped_Newton_time = np.mean( experiment_results[ "results_damped_Newton_exact_inversion" ][ epsilon ]['timings_each_step'] ), np.std( experiment_results[ "results_damped_Newton_exact_inversion" ][ epsilon ]['timings_each_step'] )
        # Plotting the time distributions along the iterations of the optimization
        plt.style.use( "default" )
        plt.rcParams.update( { 'font.size' : 12 } ) 
        plt.figure( figsize = ( 6, 4 ) )
        plt.title( r"Time distribution for each iteration, $\varepsilon=$" +str(epsilon)+ " at each iteration", fontsize = 12 )
        ## Plotting the time distribution of damped Newton
        plt.hist( experiment_results[ "results_damped_Newton_exact_inversion" ][ epsilon ]['timings_each_step'], bins = 10, color = 'lightgreen', edgecolor = 'black', alpha = 0.7 )
        plt.axvline( mean_damped_Newton_time, color = 'blue', linestyle = 'dashed', linewidth = 2, label = f'Mean: {mean_damped_Newton_time:.2f}' )
        plt.axvline( mean_damped_Newton_time - std_damped_Newton_time, color = 'gray', linestyle = 'dotted', linewidth = 2, label = f"± Std Dev: {std_damped_Newton_time:.2f}" )
        plt.axvline( mean_damped_Newton_time + std_damped_Newton_time, color = 'gray', linestyle = 'dotted', linewidth = 2 )
        plt.xlabel( "Time in ms" )
        plt.yscale( 'log' )
        plt.legend()
        plt.tight_layout()
        plt.show()
        print( "" )
# end for  

#### Hybrid optimization(Damped Newton/Sinkhorn)

In [ ]:
# Number of preconditioning eigenvectors, adjusted to be within the bounds of the array dimensions
num_eigs = 35
# Choosing the epsilon corresponding to which the Hessian is used to obtain the preconditioning vectors
preconditioning_epsilon = 0.5
null_vector, precond_vectors, _ = build_preconditioners( num_eigs, log_domainSinkhorn_Hessians[ preconditioning_epsilon ], ansatz = False )

In [ ]:
print( " Doing for (",N[0], N[1],"). " )
# Damping factor for ascent step-size
rho = 0.4
# Sufficient increase parameter in the Armijo condition 
c = 0.1
# Number of iterations
num_iterations = 50
# Inversion method
inversion_method = "exact"
# Indicator to plot time histograms
plot_time_histogram = True
f = None
for epsilon in epsilons :
    print( "For epsilon = "+str(epsilon)+":" )    
    # Initializing potential f
    if f is None:
        f = a * 0  
    print( " Iterating" )
    optimizer =  hybrid_optimization(       C,
                                            a,
                                            b,
                                            f,
                                            epsilon,
                                            rho,
                                            c,
                                            null_vector,
                                            precond_vectors[:]
                                        )    
    start = time.time()                                                  
    experiment_results[ "results_hybrid_optimization_exact_inversion" ][ epsilon ] =    optimizer._optimize(    max_iterations = num_iterations,
                                                                                                                inversion_method = inversion_method,
                                                                                                                show_inversion_timings = False,# To see timestamps at different steps during the inversion of the Hessian
                                                                                                                show_armijo_angle = False,# To see the angle between the obtained direction vector and the gradient
                                                                                                                show_condition_number = False,# To see the condition number of the matrix just before inversion
                                                                                                                observe_underflows = False# To see underflow appearing
                                                                                                        )                                                                                                            
    end = time.time() 
    # Recording time taken for each epsilon
    experiment_results[ "results_hybrid_optimization_exact_inversion" ][ epsilon ]['total_time'] = end - start
    print( "" )
    if plot_time_histogram:
        # Compute mean and standard deviation of the timings
        mean_log_domainSinkhorn_time, std_log_domainSinkhorn_time = np.mean( experiment_results[ "results_hybrid_optimization_exact_inversion" ][ epsilon ]['timings_each_step']['timings_log_domainSinkhorn'] ), np.std( experiment_results[ "results_hybrid_optimization_exact_inversion" ][ epsilon ]['timings_each_step']['timings_log_domainSinkhorn'] )
        mean_damped_Newton_time, std_damped_Newton_time = np.mean( experiment_results[ "results_hybrid_optimization_exact_inversion" ][ epsilon ]['timings_each_step']['timings_dampedNewton'] ), np.std( experiment_results[ "results_hybrid_optimization_exact_inversion" ][ epsilon ]['timings_each_step']['timings_dampedNewton'] )
        # Plotting the time distributions of the two algorithms along the iterations of the optimization
        plt.style.use( "default" )
        plt.rcParams.update( { 'font.size' : 12 } )
        fig, axes = plt.subplots( nrows = 1, ncols = 2, figsize = ( 10, 5 ), sharey = True )
        fig.suptitle( r"Time distribution for each iteration, $\varepsilon=$" +str(epsilon), fontsize = 20 )
        ## Plotting the time distribution of log-domain Sinkhorn
        axes[0].hist( experiment_results[ "results_hybrid_optimization_exact_inversion" ][ epsilon ]['timings_each_step']['timings_log_domainSinkhorn'], bins = 10, color = 'red', edgecolor = 'black', alpha = 0.7 )
        axes[0].axvline( mean_log_domainSinkhorn_time, color = 'blue', linestyle = 'dashed', linewidth = 2, label = f'Mean: {mean_log_domainSinkhorn_time:.2f}' )
        axes[0].axvline( mean_log_domainSinkhorn_time - std_log_domainSinkhorn_time, color = 'gray', linestyle = 'dotted', linewidth = 2, label = f"± Std Dev: {std_log_domainSinkhorn_time:.2f}" )
        axes[0].axvline( mean_log_domainSinkhorn_time + std_log_domainSinkhorn_time, color = 'gray', linestyle = 'dotted', linewidth = 2 )
        axes[0].set_title( "Log-domain Sinkhorn" )
        axes[0].set_xlabel( "Time in ms" )
        axes[0].set_yscale( 'log' )
        axes[0].legend()
        ## Plotting the time distribution of damped Newton
        axes[1].hist( experiment_results[ "results_hybrid_optimization_exact_inversion" ][ epsilon ]['timings_each_step']['timings_dampedNewton'], bins = 10, color = 'lightgreen', edgecolor = 'black', alpha = 0.7 )
        axes[1].axvline( mean_damped_Newton_time, color = 'blue', linestyle = 'dashed', linewidth = 2, label = f'Mean: {mean_damped_Newton_time:.2f}')
        axes[1].axvline( mean_damped_Newton_time - std_damped_Newton_time, color = 'gray', linestyle = 'dotted', linewidth = 2, label = f"± Std Dev: {std_damped_Newton_time:.2f}" )
        axes[1].axvline( mean_damped_Newton_time + std_damped_Newton_time, color = 'gray', linestyle = 'dotted', linewidth = 2 )
        axes[1].set_title( "Damped Newton" )
        axes[1].set_xlabel( "Time in ms" )
        axes[1].set_yscale( 'log' )
        axes[1].legend()
        plt.tight_layout()
        plt.show()
        print( "" )
# end for 

### ii) Using iterative method for Hessian inversion

#### Damped Newton with preconditioning

In [ ]:
# Number of preconditioning eigenvectors, adjusted to be within the bounds of the array dimensions
num_eigs = 35
# Choosing the epsilon corresponding to which the Hessian is used to obtain the preconditioning vectors
preconditioning_epsilon = 0.5
null_vector, precond_vectors, _ = build_preconditioners( num_eigs, log_domainSinkhorn_Hessians[ preconditioning_epsilon ], ansatz = False )

In [ ]:
print( " Doing for (",N[0], N[1],"). " )
# Damping factor for ascent step-size
rho = 0.2
# Sufficient increase parameter in the Armijo condition 
c = 0.1
# Absolute tolerance parameeter in iterative inversion
a_tol = 1e-12
# Relative tolerance parameter in iterative inversion
r_tol = 1e-5
# Number of iterations
num_iterations = 50
# Maximum number of iterative iversions
max_inversion = 30
# Inversion method
inversion_method = "cg"
# Indicator to plot time histograms
plot_time_histogram = False
f = None
for epsilon in epsilons :
    print( "For epsilon = "+str(epsilon)+":" )    
     # Initializing potential f
    if f is None:
        f = a * 0 
    print( " Iterating" )
    optimizer =  semi_dual_damped_Newton_with_preconditioning(  C,
                                                                a,
                                                                b,
                                                                f,
                                                                epsilon,
                                                                rho,
                                                                c,
                                                                null_vector,
                                                                precond_vectors[:]
                                                            )
    start = time.time()                                                      
    experiment_results[ "results damped Newton_iterative_inversion" ][ epsilon ] = optimizer._optimize(     max_iterations = num_iterations,
                                                                                                            inversion_method = inversion_method,
                                                                                                            max_inversions = max_inversion,
                                                                                                            relative_tol = r_tol,
                                                                                                            absolute_tol = a_tol,
                                                                                                            show_inversion_timings = False,# To see timestamps at different steps during the inversion of the Hessian
                                                                                                            show_armijo_angle = False,# To see the angle between the obtained direction vector and the gradient
                                                                                                            show_condition_number = False,# To see the condition number of the matrix just before inversion
                                                                                                            observe_underflows = False# To see underflow appearing
                                                                                                        )
    end = time.time() 
    # Recording time taken for each epsilon
    experiment_results[ "results damped Newton_iterative_inversion" ][ epsilon ]['total_time'] = end - start
    print( "" )
    if plot_time_histogram:
        # Compute mean and standard deviation of the timings
        mean_damped_Newton_time, std_damped_Newton_time = np.mean( experiment_results[ "results damped Newton_iterative_inversion" ][ epsilon ]['timings_each_step'] ), np.std( experiment_results[ "results damped Newton_iterative_inversion" ][ epsilon ]['timings_each_step'] )
        # Plotting the time distributions along the iterations of the optimization
        plt.style.use( "default" )
        plt.rcParams.update( { 'font.size' : 12 } ) 
        plt.figure( figsize = ( 6, 4 ) )
        ## Plotting the time distribution of damped Newton
        plt.title( r"Time distribution for each iteration, $\varepsilon=$" +str(epsilon)+ " at each iteration", fontsize = 12 )
        plt.hist( experiment_results[ "results damped Newton_iterative_inversion" ][ epsilon ]['timings_each_step'], bins = 10, color = 'lightgreen', edgecolor = 'black', alpha = 0.7 )
        plt.axvline( mean_damped_Newton_time, color = 'blue', linestyle = 'dashed', linewidth = 2, label = f'Mean: {mean_damped_Newton_time:.2f}' )
        plt.axvline( mean_damped_Newton_time - std_damped_Newton_time, color = 'gray', linestyle = 'dotted', linewidth = 2, label = f"± Std Dev: {std_damped_Newton_time:.2f}" )
        plt.axvline( mean_damped_Newton_time + std_damped_Newton_time, color = 'gray', linestyle = 'dotted', linewidth = 2 )
        plt.xlabel( "Time in ms" )
        plt.yscale( 'log' )
        plt.legend()
        plt.tight_layout()
        plt.show()
        print( "" )
# end for       

#### Hybrid optimization(Damped Newton/Sinkhorn)

In [ ]:
# Number of preconditioning eigenvectors, adjusted to be within the bounds of the array dimensions
num_eigs = 35
# Choosing the epsilon corresponding to which the Hessian is used to obtain the preconditioning vectors
preconditioning_epsilon = 0.5
null_vector, precond_vectors, _ = build_preconditioners( num_eigs, log_domainSinkhorn_Hessians[ preconditioning_epsilon ], ansatz = False )

In [ ]:
print( " Doing for (",N[0], N[1],"). " )
# Damping factor for ascent step-size
rho = 0.2
# Sufficient increase parameter in the Armijo condition 
c = 0.1
# Absolute tolerance parameeter in iterative inversion
a_tol = 1e-12
# Relative tolerance parameter in iterative inversion
r_tol = 1e-5
# Number of iterations
num_iterations = 50
# Maximum number of iterative iversions
max_inversion = 30
# Inversion method
inversion_method = "cg"
# Indicator to plot time histograms
plot_time_histogram = True
f = None
for epsilon in epsilons :
    print( "For epsilon = "+str(epsilon)+":" )    
     # Initializing potential f
    if f is None:      
        f = a * 0 
    print( " Iterating" )
    optimizer =  hybrid_optimization(   C,
                                        a,
                                        b,
                                        f,
                                        epsilon,
                                        rho,
                                        c,
                                        null_vector,
                                        precond_vectors[:]
                                    )
    start = time.time()                                                      
    experiment_results[ "results_hybrid_optimization_iterative_inversion" ][ epsilon ] = optimizer._optimize(   max_iterations = num_iterations,
                                                                                                                inversion_method = inversion_method,
                                                                                                                max_inversions = max_inversion,
                                                                                                                relative_tol = r_tol,
                                                                                                                absolute_tol = a_tol,
                                                                                                                show_inversion_timings = False,# To see timestamps at different steps during the inversion of the Hessian
                                                                                                                show_armijo_angle = False,# To see the angle between the obtained direction vector and the gradient
                                                                                                                show_condition_number = False,# To see the condition number of the matrix just before inversion
                                                                                                                observe_underflows = False# To see underflow appearing
                                                                                                            )
    end = time.time() 
    # Recording time taken for each epsilon
    experiment_results[ "results_hybrid_optimization_iterative_inversion" ][ epsilon ]['total_time'] = end - start
    print( "" )
    if plot_time_histogram:
        # Compute mean and standard deviation of the timings
        mean_log_domainSinkhorn_time, std_log_domainSinkhorn_time = np.mean( experiment_results[ "results_hybrid_optimization_iterative_inversion" ] [ epsilon ]['timings_each_step']['timings_log_domainSinkhorn'] ), np.std( experiment_results[ "results_hybrid_optimization_iterative_inversion" ] [ epsilon ]['timings_each_step']['timings_log_domainSinkhorn'] )
        mean_damped_Newton_time, std_damped_Newton_time = np.mean( experiment_results[ "results_hybrid_optimization_iterative_inversion" ] [ epsilon ]['timings_each_step']['timings_dampedNewton'] ), np.std( experiment_results[ "results_hybrid_optimization_iterative_inversion" ] [ epsilon ]['timings_each_step']['timings_dampedNewton'] )
        # Plotting the time distributions of the two algorithms along the iterations of the optimization
        plt.style.use( "default" )
        plt.rcParams.update( { 'font.size' : 12 } )
        fig, axes = plt.subplots( nrows = 1, ncols = 2, figsize = ( 10, 5 ), sharey = True )
        fig.suptitle( r"Time distribution for each iteration, $\varepsilon=$" +str(epsilon), fontsize = 20 )
        ## Plotting the time distribution of log-domain Sinkhorn
        axes[0].hist( experiment_results[ "results_hybrid_optimization_iterative_inversion" ][ epsilon ]['timings_each_step']['timings_log_domainSinkhorn'], bins = 10, color = 'red', edgecolor = 'black', alpha = 0.7 )
        axes[0].axvline( mean_log_domainSinkhorn_time, color = 'blue', linestyle = 'dashed', linewidth = 2, label = f'Mean: {mean_log_domainSinkhorn_time:.2f}' )
        axes[0].axvline( mean_log_domainSinkhorn_time - std_log_domainSinkhorn_time, color = 'gray', linestyle = 'dotted', linewidth = 2, label = f"± Std Dev: {std_log_domainSinkhorn_time:.2f}" )
        axes[0].axvline( mean_log_domainSinkhorn_time + std_log_domainSinkhorn_time, color = 'gray', linestyle = 'dotted', linewidth = 2 )
        axes[0].set_title( "Log-domain Sinkhorn" )
        axes[0].set_xlabel( "Time in ms" )
        axes[0].set_yscale( 'log' )
        axes[0].legend()
        ## Plotting the time distribution of damped Newton
        axes[1].hist( experiment_results[ "results_hybrid_optimization_iterative_inversion" ][ epsilon ]['timings_each_step']['timings_dampedNewton'], bins = 10, color = 'lightgreen', edgecolor = 'black', alpha = 0.7 )
        axes[1].axvline( mean_damped_Newton_time, color = 'blue', linestyle = 'dashed', linewidth = 2, label = f'Mean: {mean_damped_Newton_time:.2f}')
        axes[1].axvline( mean_damped_Newton_time - std_damped_Newton_time, color = 'gray', linestyle = 'dotted', linewidth = 2, label = f"± Std Dev: {std_damped_Newton_time:.2f}" )
        axes[1].axvline( mean_damped_Newton_time + std_damped_Newton_time, color = 'gray', linestyle = 'dotted', linewidth = 2 )
        axes[1].set_title( "Damped Newton" )
        axes[1].set_xlabel( "Time in ms" )
        axes[1].set_yscale( 'log' )
        axes[1].legend()
        plt.tight_layout()
        plt.show()
        print( "" )
# end for   

### Error plot

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )
plt.style.use( "seaborn-v0_8-notebook" )
fig, axes = plt.subplots( nrows = 4, ncols = 1, figsize = ( 20, 30 ), sharex = True )
if inversion_method == 'cg':
    method_text = "conjugate gradient"
else:   
    method_text = "GMRES"
fig.suptitle( r"$E_{\alpha,\beta}\left(P_{\varepsilon}\right) = \|P_{\varepsilon}\mathbb{1}_{m} -\alpha\|_1+\|P_{\varepsilon}^{T}\mathbb{1}_{n} -\beta\|_1$", y = .92, fontsize = 20, fontweight = 'bold'  )
axes[0].set_title( "Damped Newton with preconditioning and using exact inversion", fontsize = 15 )
for epsilon in experiment_results[ "results_damped_Newton_exact_inversion" ].keys():
    axes[0].plot( experiment_results[ "results_damped_Newton_exact_inversion" ][epsilon]['errors'],  marker = 'o', label = r'For $\varepsilon = $'+str(epsilon) )
# end for
axes[0].set_ylabel( r"$E_{\alpha,\beta}\left(P_{\varepsilon}\right)$" )
axes[0].legend( loc = "upper right", fontsize = "large" )    
axes[0].set_yscale( 'log' ) 
axes[1].set_title( "Hybrid optimization(Damped Newton/Sinkhorn) using exact inversion", fontsize = 15 ) 
for epsilon in experiment_results[ "results_hybrid_optimization_exact_inversion" ].keys():
    colors = []
    errors = []
    for i in range( len( experiment_results[ "results_hybrid_optimization_exact_inversion" ][epsilon]['errors'] ) ):
        errors.append( experiment_results[ "results_hybrid_optimization_exact_inversion" ][epsilon]['errors'][i] )
        if experiment_results[ "results_hybrid_optimization_exact_inversion" ][epsilon]['update_indicator'][i] == 'log-domain Sinkhorn':
            colors.append( 'red' )# If log-domain Sinkhorn update was used at this step
        else:
            colors.append( "lightgreen" )# If damped Newton was used at this step
    # end for
    axes[1].plot( errors,  marker = 'o', label = r'For $\varepsilon = $'+str(epsilon) )
    # Plot each marker with a different color
    for i in range( len( colors ) ):
        axes[1].scatter( i, errors[i], color = colors[i], s = 50, edgecolors = 'black', zorder = 2 )
    # end for
# end for
# Add legend entries for the markers (plot dummy points for legend)
axes[1].scatter( [], [], color = 'red', s = 50, edgecolors = 'black', label = 'Log-domain Sinkhorn' )# Dummy red marker
axes[1].scatter( [], [], color = "lightgreen", s = 50, edgecolors = 'black', label = 'Damped Newton' )# Dummy green marker
axes[1].set_ylabel( r"$E_{\alpha,\beta}\left(P_{\varepsilon}\right)$" )
axes[1].legend( loc = "upper right", fontsize = "large" )    
axes[1].set_yscale( 'log' )  
axes[2].set_title( "Damped Newton with preconditioning and using " +method_text+ " for inversion", fontsize = 15 )
for epsilon in  experiment_results[ "results damped Newton_iterative_inversion" ].keys():
    axes[2].plot(  experiment_results[ "results damped Newton_iterative_inversion" ][epsilon]['errors'],  marker = 'o', label = r'For $\varepsilon = $'+str(epsilon) )
# end for
axes[2].set_ylabel( r"$E_{\alpha,\beta}\left(P_{\varepsilon}\right)$" )
axes[2].legend( loc = "upper right", fontsize = "large" )    
axes[2].set_yscale( 'log' ) 
axes[3].set_title( "Hybrid optimization(Damped Newton/Sinkhorn) using " +method_text+ " for inversion", fontsize = 15 ) 
for epsilon in experiment_results[ "results_hybrid_optimization_iterative_inversion" ] .keys():
    colors = []
    errors = []
    for i in range( len( experiment_results[ "results_hybrid_optimization_iterative_inversion" ] [epsilon]['errors'] ) ):
        errors.append( experiment_results[ "results_hybrid_optimization_iterative_inversion" ] [epsilon]['errors'][i] )
        if experiment_results[ "results_hybrid_optimization_iterative_inversion" ] [epsilon]['update_indicator'][i] == 'log-domain Sinkhorn':
            colors.append( 'red' )# If log-domain Sinkhorn update was used at this step
        else:
            colors.append( "lightgreen" )# If damped Newton was used at this step
    # end for
    axes[3].plot( errors,  marker = 'o', label = r'For $\varepsilon = $'+str(epsilon) )
    # Plot each marker with a different color
    for i in range( len( colors ) ):
        axes[3].scatter( i, errors[i], color = colors[i], s = 50, edgecolors = 'black', zorder = 2 )
    # end for
# end for
# Add legend entries for the markers (plot dummy points for legend)
axes[3].scatter( [], [], color = 'red', s = 50, edgecolors = 'black', label = 'Log-domain Sinkhorn' )# Dummy red marker
axes[3].scatter( [], [], color = "lightgreen", s = 50, edgecolors = 'black', label = 'Damped Newton' )# Dummy green marker
axes[3].set_ylabel( r"$E_{\alpha,\beta}\left(P_{\varepsilon}\right)$" )
axes[3].legend( loc = "upper right", fontsize = "large" )    
axes[3].set_yscale( 'log' )  
plt.xlabel( 'Number of iterations' )
plt.savefig( image_folder_path + "/Error_plot_pure_dampedNewton_vs_hybrid_optimization.pdf", format = 'pdf' )
plt.show() 

#### Comment

##### From the above two comparisons between the damped Newton with preconditioning and iterative inversion, we can see that having the option of choosing updates of log-domain Sinkhorn has a stabilizing effect. In the case of exact inversion it simply enables convergence for small $\varepsilon$ while in the case where we use iterative inversion it reduces the number of iterations required to converge.

## Time comparison between the use of log-domain Sinkhorn and hybrid(damped Newton/Sinkhorn) optimization with exact and iterative methods for inversion.

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )
plt.style.use( "seaborn-v0_8-notebook" )
plt.figure( figsize = ( 20, 7 ) )   
plt.title( "Time plot", fontsize = 15 )
log_domnainSinkhorn_timings = [ experiment_results[ "results_log_domain_Sinkhorn" ][epsilon]['total_time' ] for epsilon in epsilons ]
hybrid_optimization_exact_inversion_timings = [ experiment_results[ "results_hybrid_optimization_exact_inversion" ][epsilon]['total_time' ] for epsilon in epsilons ]
hybrid_optimization_iterative_inversion_timings = [ experiment_results[ "results_hybrid_optimization_iterative_inversion" ][epsilon]['total_time' ] for epsilon in epsilons ]
plt.plot( epsilons[::-1], log_domnainSinkhorn_timings[::-1], label = "Log-domain Sinkhorn",  linewidth = 2, marker = 'o' ) 
plt.plot( epsilons[::-1], hybrid_optimization_exact_inversion_timings[::-1], label = "Hybrid optimization with exact method for Hessian inversion",  linewidth = 2, marker = 'o' ) 
plt.plot( epsilons[::-1], hybrid_optimization_iterative_inversion_timings[::-1], label = "Hybrid optimization with iterative method for Hessian inversion",  linewidth = 2, marker = 'o' ) 
plt.xlabel( r"$\varepsilon$" )  
plt.ylabel( "Time in seconds" ) 
plt.legend( loc = "upper right", fontsize = "large" )
plt.yscale( 'log' ) 
plt.xscale( 'log' )                                                                                                                                                                                                                                                              
plt.savefig( image_folder_path + "/Timeplot_iterative_optimization_vs_logdomainSinkhorn.pdf", format = 'pdf' ) 
plt.show()                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

## Diagnostics

### I. Instability in the Hessian due to underflow when $\varepsilon$ is small

Here we use the damped Newton with preconditioning to observe the instabilities that arise due to underflows by looking into:

i) Angle betweeen the direction vector and the gradient

ii) 10 smallest and 10 largest eigenvalues

iii) Conition number

iv) Underflow: Observing values less than the system threshold to be considered as $0$

In [ ]:
# Number of preconditioning eigenvectors, adjusted to be within the bounds of the array dimensions
num_eigs = 35
# Choosing the epsilon corresponding to which the Hessian is used to obtain the preconditioning vectors
preconditioning_epsilon = 0.5
null_vector, precond_vectors, _ = build_preconditioners( num_eigs, log_domainSinkhorn_Hessians[ preconditioning_epsilon ], ansatz = False )

#### Testing using exact inversion

In [ ]:
print( " Doing for (",N[0], N[1],"). " )
# Damping factor for ascent step-size
rho = 0.4
# Sufficient increase parameter in the Armijo condition 
c = 0.1
# Number of iteratis
num_iterations = 5
# Inversion method
inv_method = "exact"
results_semi_dual_damped_Newton_exact_inversion = {}
f = None
for epsilon in [ 0.05, 0.01, 0.005, 0.0009, 0.0007, 0.0005 ] :
    print( "For epsilon = "+str(epsilon)+":" )    
    # Initializing potential f
    if f is None:
        f = a * 0  
    print( " Iterating" )
    optimizer =  semi_dual_damped_Newton_with_preconditioning(  C,
                                                                a,
                                                                b,
                                                                f,
                                                                epsilon,
                                                                rho,
                                                                c,
                                                                null_vector,
                                                                precond_vectors[:]
                                                            )    
    start = time.time()                                                  
    results_semi_dual_damped_Newton_exact_inversion[ epsilon ] = optimizer._optimize(   max_iterations = num_iterations,
                                                                                        inversion_method = inv_method,
                                                                                        show_armijo_angle = True,# To see the angle between the obtained direction vector and the gradient
                                                                                        show_condition_number = True,# To see the condition number of the matrix just before inversion
                                                                                        observe_underflows = True# To see underflow appearing
                                                            )
    end = time.time() 
    # Recording time taken for each epsilon
    results_semi_dual_damped_Newton_exact_inversion[ epsilon ]['total_time'] = end - start
    print( "" )
# end for  

##### Comment
Here as we can see that the angle between the direction vector and the gradient is negative for small $\varepsilon$. In the setup of maximization, slope should be positive for the objective function to increase in the direction.

#### Testing using iterative inversion

In [ ]:
print( " Doing for (",N[0], N[1],"). " )
# Damping factor for ascent step-size
rho = 0.4
# Sufficient increase parameter in the Armijo condition 
c = 0.1
# Number of iteratis
num_iterations = 5
# Inversion method
inv_method = "cg"
results_semi_dual_damped_Newton_iterative_inversion = {}
f = None
for epsilon in [ 0.05, 0.01, 0.005, 0.0009, 0.0007, 0.0005 ] :
    print( "For epsilon = "+str(epsilon)+":" )    
    # Initializing potential f
    if f is None:
        f = a * 0  
    print( " Iterating" )
    optimizer =  semi_dual_damped_Newton_with_preconditioning(  C,
                                                                a,
                                                                b,
                                                                f,
                                                                epsilon,
                                                                rho,
                                                                c,
                                                                null_vector,
                                                                precond_vectors[:]
                                                            )    
    start = time.time()                                                  
    results_semi_dual_damped_Newton_iterative_inversion[ epsilon ] = optimizer._optimize(   max_iterations = num_iterations,
                                                                                            inversion_method = inv_method,
                                                                                            show_armijo_angle = True,# To see the angle between the obtained direction vector and the gradient
                                                                                            show_condition_number = True,# To see the condition number of the matrix just before inversion
                                                                                            observe_underflows = True# To see underflow appearing
                                                                                        )
    end = time.time() 
    # Recording time taken for each epsilon
    results_semi_dual_damped_Newton_iterative_inversion[ epsilon ]['total_time'] = end - start
    print( "" )
# end for  

### II. Role of the parameters $c$ and $\rho$ in the hybrid algorithm
Here we look at the effect the sufficient increase parametter $c$ and the damping factor $\rho$ has on the performance of the hybrid(damped Newton/Sinkhorn) optimization.

In [ ]:
# Taking some variations of the tuple (c, rho)
c_rho = [ ( 0.1, 0.2 ), ( 0.1, 0.4 ), ( 0.1, 0.6 ), ( 0.1, 0.9 ), ( 0.1, 0.4 ), ( 0.2, 0.4 ), ( 0.6, 0.4 ), ( 0.8, 0.4 ) ]
# Number of iteratis
num_iterations = 50
# Inversion method
inv_method = "exact"
f = None
epsilon = 0.0007
out_semi_dual_damped_Newton_exact = {}
for tuple in c_rho:
    # Initializing potential f
    if f is None:
        f = a * 0  
    print("For c = "+str(tuple[0])+" and rho = "+str(tuple[1])+":")
    print( " Iterating" )
    optimizer =  hybrid_optimization(   C,
                                        a,
                                        b,
                                        f,
                                        epsilon,
                                        tuple[1],
                                        tuple[0],
                                        null_vector,
                                        precond_vectors[:]
                                    )    
    out_semi_dual_damped_Newton_exact[ tuple ] = optimizer._optimize(   max_iterations = num_iterations,
                                                                        inversion_method = inv_method,
                                                                    )           
    print( "" )
# end for

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )
plt.style.use( "seaborn-v0_8-notebook" )
plt.figure( figsize = ( 20, 7 ) )
plt.title( r"$E_{\alpha,\beta}\left(P_{\varepsilon}\right) = \|P_{\varepsilon}\mathbb{1}_{m} -\alpha\|_1+\|P_{\varepsilon}^{T}\mathbb{1}_{n} -\beta\|_1,\ \varepsilon = $"+str(epsilon) ) 
for tuple in c_rho:
  errors = np.asarray( out_semi_dual_damped_Newton_exact[tuple]['errors'] )
  plt.plot( errors, label = r'Damped Newton with preconditioning for $(c,\ \rho) = $' + str(tuple), linewidth = 2 )
# end for
plt.xlabel( " Number of iterations " )
plt.ylabel( r"$E_{\alpha,\beta}\left(P_{\varepsilon}\right)$" )
plt.yscale( 'log' )
plt.legend( loc = "upper right", fontsize = "large" )
plt.show()

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )
plt.style.use( "seaborn-v0_8-notebook" )
plt.figure( figsize = ( 20, 7 ) )
fig, axes = plt.subplots( 4, 1, figsize = ( 15, 15 ) )  
axes[0].set_title( r'Backtracking time plot for varying value of $\rho$' )
for j in range( 4 ) :
  n = len( out_semi_dual_damped_Newton_exact[ c_rho[j] ][ 'backtracking_info' ] )
  timings = []
  for i in range( n ):
    timings.append( np.asarray( out_semi_dual_damped_Newton_exact[ c_rho[j] ][ 'backtracking_info' ][i][ 'time' ] ) )
  axes[0].plot( timings, label = r'c=' +str(c_rho[j][0])+ r', $\rho=$' +str(c_rho[j][1]), linewidth = 2, marker = 'o' )
# end for
axes[0].set_xlabel( " Number of iterations " )
axes[0].set_ylabel( "Time in ms" )
axes[0].set_yscale( 'log' )
axes[0].legend( loc = "upper right", fontsize = "large" )
axes[1].set_title( r'Backtracking time plot for varying value of $c$' )
for j in range( 4, 8 ) :
  n = len( out_semi_dual_damped_Newton_exact[ c_rho[j] ][ 'backtracking_info' ] )
  timings = []
  for i in range( n ):
    timings.append( np.asarray( out_semi_dual_damped_Newton_exact[ c_rho[j] ][ 'backtracking_info' ][i][ 'time' ] ) )
  axes[1].plot( timings, label = r'c=' +str(c_rho[j][0])+ r', $\rho=$' +str( c_rho[j][1]), linewidth = 2, marker = 'o' )
# end for
axes[1].set_xlabel( " Number of iterations " )
axes[1].set_ylabel( "Time in ms" )
axes[1].set_yscale( 'log' )
axes[1].legend( loc = "upper right", fontsize = "large" )
axes[2].set_title( r'Reduction count plot for varying value of $\rho$' )
for j in range( 4 ) :
  n = len( out_semi_dual_damped_Newton_exact[ c_rho[j] ][ 'backtracking_info' ] )
  reduction_counts = []
  for i in range( n ):
    reduction_counts.append( np.asarray( out_semi_dual_damped_Newton_exact[ c_rho[j] ][ 'backtracking_info' ][i][ 'reduction_count' ] ) )
  axes[2].plot( reduction_counts, label = r'c=' +str(c_rho[j][0])+ r', $\rho=$' +str( c_rho[j][1]), linewidth = 2, marker = 'o' )
# end for
axes[2].set_xlabel( " Number of iterations " )
axes[2].set_ylabel( "Reduction count" )
axes[2].legend( loc = "upper right", fontsize = "large" )
axes[3].set_title( r'Damping count plot for varying value of $c$' )
for j in range( 4, 8 ) :
  n = len( out_semi_dual_damped_Newton_exact[ c_rho[j] ][ 'backtracking_info' ] )
  reduction_counts = []
  for i in range( n ):
    reduction_counts.append( np.asarray( out_semi_dual_damped_Newton_exact[ c_rho[j] ][ 'backtracking_info' ][i][ 'reduction_count' ] ) )
  axes[3].plot( reduction_counts, label = r'c=' +str(c_rho[j][0])+ r', $\rho=$' +str( c_rho[j][1]), linewidth = 2, marker = 'o' )
# end for
axes[3].set_xlabel( " Number of iterations " )
axes[3].set_ylabel( "Reduction count" )
axes[3].legend( loc = "upper right", fontsize = "large" )
plt.tight_layout()
plt.show()